# Mask R-CNN ResNet-50 FPN v2 (COCO) — DIMER instance segmentation and bounded fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/maskrcnn-instance-segmentation-pipeline/blob/main/tutorials/maskrcnn_instance_segmentation_colab.ipynb) [![torchvision](https://img.shields.io/badge/torchvision-maskrcnn__resnet50__fpn__v2-ee4c2c?style=flat&logo=pytorch&logoColor=white)](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn_v2.html) [![Upstream](https://img.shields.io/badge/Upstream-pytorch%2Fvision-181717?style=flat&logo=github&logoColor=white)](https://github.com/pytorch/vision) [![arXiv](https://img.shields.io/badge/arXiv-1703.06870-b31b1b.svg)](https://arxiv.org/abs/1703.06870) [![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** instance segmentation over the COCO classes with torchvision's Mask R-CNN ResNet-50 FPN v2, and a bounded fine-tune that re-heads the box and mask predictors onto your own class vocabulary, evaluates the result on a held-out split with box and mask average precision, and exports a reloadable SafeTensors adapter

**This notebook is standalone.** It carries the repository's package (2 modules under `src/maskrcnn_instance_segmentation_pipeline/`, at revision `6244270d0ffb`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and `download.pytorch.org` for the checkpoint file pinned by SHA-256 `73cbd0190fcbe3ba339921fbce2c3a0b6bb9126c9a133c85e43a2a8e060a109e` (~186 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.1); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (the kernel's own packages are left alone, so no restart is needed and Run all completes in one pass), stages and digest-verifies the pinned checkpoint, runs COCO instance segmentation on a drawn scene, validates the sign dataset, splits it into training and held-out parts, measures the pre-adaptation baseline, **runs the bounded fine-tune**, re-evaluates on the held-out split, segments unseen images, exports the adapter, reloads it onto a fresh base model to verify the instances, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag, and no clone, DIMER worker, credential, upload dialog or configuration edit is required (NOTEBOOK_SPEC 2.2 §5, RUN7, FT2). CUDA is used when present; a GPU runtime is recommended. Measured times are listed under Prerequisites.

**Bring Your Own Data:** Two optional BYOD branches are included, and both are off by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own image through the same validation, segmentation and evaluation-report stages as the sample scene and writes its input manifest. `USE_BYOD_DATASET` takes your own labelled instance masks through the full adaptation workflow — validate, split, baseline, fine-tune, evaluate, export, reload and compare — under NOTEBOOK_SPEC 2.2 DAT14, and writes its own result JSON and detections CSV. Set `BYOD_IMAGE_PATH` or `BYOD_DATASET_DIR` to read from a location without an upload dialog (EXE2).

Mask R-CNN is a two-stage instance segmentation model. A ResNet-50 backbone with a feature pyramid network (FPN) produces feature maps at five scales; a region proposal network (RPN) proposes candidate boxes; and for each surviving region the ROI heads predict a class, a refined box, and a 28×28 mask that is pasted back into the image. This notebook uses torchvision's **v2** weights (`MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1`): torchvision describes the v2 builder as the improved Mask R-CNN from the *Benchmarking Detection Transfer Learning with Vision Transformers* paper, trained with an enhanced recipe, and reports 47.4 box AP and 41.8 mask AP on COCO val2017 for these weights. The model's own transform resizes each image so its shorter side is 800 px, and every instance whose class probability reaches a caller-owned threshold is returned as an xyxy box plus a boolean mask in input pixels.

**The default path really adapts the model:** it re-heads Mask R-CNN onto a three-class traffic sign vocabulary that does not exist in COCO (`stop-sign`, `yield-sign`, `speed-limit-sign`), measures a pre-adaptation baseline, runs a bounded fine-tune with the ResNet-50 body frozen, scores the result on a held-out split with box and mask average precision (AP@[.50:.95] and AP50), runs the adapted model on unseen images, exports the changed tensors as a SafeTensors adapter, and reloads that adapter onto a fresh copy of the verified base model to check that it reproduces the same instances. Every number you see is measured in this notebook runtime. One result is worth knowing in advance: on these drawn signs the adapted model reaches the **ceiling** of most held-out metrics — in the recorded run every mask AP row read 1.0 — so this notebook teaches the adaptation *workflow* and how to read a saturated result, not the size of a gain. Section 9 explains what a 1.0 row does and does not show, and Section 14 compares settings on the one held-out number that still had headroom (box AP@[.50:.95]).

**Who this is for.** A learner who knows basic Python and PIL, has met bounding boxes and intersection-over-union, and wants to see how a pretrained instance segmentation model is re-headed and fine-tuned for a new class vocabulary, and how the result is measured without fooling themselves. No prior experience with Mask R-CNN or fine-tuning is assumed; *instance*, *mask*, *NMS*, *re-heading* and *average precision* are explained where they are first used and again in the **Glossary** at the end.

**Input → Model → Output.**

| | Inference | Adaptation |
|---|---|---|
| Input | one RGB image, sides 16..4,096 px, and a score threshold you choose | drawn training images, each with xyxy pixel boxes, one boolean mask per instance and labels from a three-class sign vocabulary |
| Model | Mask R-CNN ResNet-50 FPN v2 with its 91-slot COCO head | the same model with fresh box and mask predictors for the three classes; FPN, RPN and ROI heads trained, ResNet-50 body frozen |
| Output | instances (label, softmax score, xyxy box, boolean mask) at or above the threshold, at most `MAX_DETECTIONS` (100) | held-out box and mask AP before and after, instances on unseen images, and a SafeTensors adapter that reloads to the same instances |

**How to use this notebook.** Choose a runtime (a GPU runtime such as a Colab T4 is much faster; CPU works, slowly — see Prerequisites), then **Runtime → Run all**. Run all completes in one pass: Section 1 builds a separate locked environment and installs nothing into the notebook's own Python, so no restart is needed. Sections 1–3 are **infrastructure** — the isolated environment, the carried code (collapsed) and the model verification — and can be run without study. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited; the defaults reproduce the recorded path. Every training setting (`EPOCHS`, `LEARNING_RATE`, `BATCH_SIZE`, `FREEZE_BACKBONE`) sits in the Section 8 cell, and Section 8 always starts from the freshly re-headed model (`reset_to_pretrained()`), so re-running it after a change compares the new setting against the same starting point. Before each of Sections 4–11 runs you are asked to **predict** what they will print; the following section opens with **What to notice** and a collapsible **Check your reasoning** block with a worked answer from a recorded run. Section 14 is a **change-one-thing activity**; it and each optional experiment name the cell to change and the cell to re-run from (**Runtime → Run after**). **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Your notes are optional and are not required submissions. Two environment variables change how Section 1 behaves, and neither is needed on Colab or Kaggle: `DIMER_NOTEBOOK_CI_PREINSTALLED=1` (set by an executor that has already installed exactly these pins) skips the isolated environment and runs every cell in the kernel, and Section 1 prints that it did so; `DIMER_ISOLATED_ENV` names the folder the environment is built in (default `dimer_isolated_env`).

**Roadmap:** 1–3 infrastructure → 4 the pretrained model on a drawn scene *(scores, thresholds, box and mask IoU)* → 5 blank and noise probes *(evaluation practice)* → 6 build and validate the sign dataset → 7 split, re-head and measure the baseline → 8 bounded fine-tune → 9 held-out box and mask AP, and what a 1.0 row means → 10 unseen images → 11 export and reload the adapter → 12 outputs → 13 optional BYOD → 14 **change one thing: unfreeze the backbone** → conclude.

**Learning objectives:** by the end you should be able to (1) describe the path *image → ResNet-50 + FPN features → region proposals → per-region class, box and 28×28 mask* and say why a threshold decides what is returned (Section 4); (2) explain why an instance on a blank or noise image is a false positive, and how instances that appear only at the evaluation threshold enter average precision (Section 5); (3) explain why a new vocabulary needs new box and mask predictors and why their baseline is near zero (Sections 6–7); (4) read a training loss as optimisation evidence and held-out box and mask AP as task evidence (Sections 8–9); (5) identify a saturated metric — a 1.0 row — and explain why it cannot rank two settings (Section 9); (6) check that an exported adapter reproduces the evaluated model (Section 11); and (7) predict and then measure what unfreezing the backbone changes when both runs start from the same re-headed model (Section 14). Along the way the notebook builds a locked runtime, stages and digest-verifies the pinned checkpoint, and writes machine-readable outputs with provenance.

**This notebook does not demonstrate:** real-world traffic sign segmentation (the adaptation dataset is drawn in code, so the model learns these renderings and nothing about road photographs); COCO benchmark results (the average-precision helper here is a compact implementation without pycocotools area ranges or crowd handling, and it is run on synthetic data only); full-schedule Mask R-CNN training (the v2 weights come from a long multi-GPU recipe, recorded in torchvision's weight metadata as pytorch/vision pull request 5773; the tutorial runs a few epochs on a few dozen drawn images); score calibration or a deployment threshold for the adapted model; semantic or panoptic segmentation; keypoints; video tracking.

## Prerequisites

- **Learner:** basic Python and PIL, and Colab or Jupyter familiarity; no prior experience with instance segmentation models or fine-tuning. The notebook explains region proposals, softmax scores after non-maximum suppression, masks, re-heading, AP50 and AP@[.50:.95] where they are first used; the Glossary repeats them.
- **Runtime:** a fresh supported runtime (Google Colab, Kaggle or Linux Jupyter — **Linux x86_64 only**; the notebook builds its own isolated Python 3.12.12 environment from manylinux wheels, so a Windows or macOS kernel is not supported). A CUDA GPU such as a Colab or Kaggle T4 is recommended for the fine-tune and is used automatically when present. Measured, each figure with its environment: a Kaggle Tesla T4 run of the earlier in-kernel-install revision (notebook blob `a8f04d11`, 2026-09-25, every field at its default) took 486.2 s in two passes — 377.1 s until the install cell stopped for a restart, then 109.1 s for every cell after the restart, fine-tune included; a local CPU run of this revision's learner cells at a **reduced** configuration (24-thread Windows workstation, 2026-10-06, install skipped, checkpoint pre-staged, `N_IMAGES = 8`, `EPOCHS = 1`) took 54.9 s of cell time, 5.7 s per training step of 2 images (the 2026-10-03 review measured about 30 s per step on the same workstation while it was shared with other jobs). **Estimate, not a measurement:** at that step time the default fine-tune (`EPOCHS = 10` over 30 images, 150 steps) would take about 15 minutes on such a CPU when it is otherwise idle, and over an hour at the slower step time, so use a GPU for the default path. This revision's isolated-environment build has not yet been timed on a hosted runtime; expect it to add several minutes (an estimate). The locked install (PyTorch 2.14.0 with its CUDA libraries) and the ~186 MB checkpoint are the largest downloads.
- **Knowledge:** basic Python, PIL and NumPy; bounding boxes as xyxy pixel coordinates; binary masks as boolean arrays; intersection-over-union (IoU) for boxes and for masks; and how to read average precision (AP50 and AP@[.50:.95]).
- **Data:** the default path generates everything in code with `samples.py` and downloads no dataset: one 640×480 COCO demonstration scene and a labelled sign dataset (`N_IMAGES`, default 40 images), each object with an exact mask. BYOD is optional and off by default. Expected BYOD input: one image, or a directory holding `annotations.json` — a list of `{'file': 'name.png', 'boxes': [[x0, y0, x1, y1], ...], 'labels': [name, ...], 'masks': ['name_mask0.png', ...]}` objects with boxes in **pixel** coordinates — and the image and mask files it names (one single-channel PNG per instance, non-zero pixels inside the instance). Through the upload dialog, upload `annotations.json`, the images and the masks as **flat files (no folders)**.
- **Privacy:** Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API. The default path uploads nothing.
- **External access:** `download.pytorch.org` only, to fetch the pinned `torchvision/maskrcnn_resnet50_fpn_v2` checkpoint (~186 MB) at revision `73cbd0190fcb…`. No repository access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 33 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab import some packages, NumPy among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 33 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins below) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message. The printed dictionary names the isolated Python version and the kernel's.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '7cfc22f2db3a77198e3732ca67f8f00bcd81f42bcead81c6ad638dd266d4f5ff'
LOCKED_PACKAGES = 33
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml -c <detr-detection-pipeline c3c81cb lock versions> --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
# Direct pins: the pyproject.toml dependencies at this revision, unchanged (torch 2.14.0, torchvision 0.29.0, safetensors 0.8.0,
# numpy 2.5.3, pillow 11.3.0). Every transitive version is constrained to the lock of detr-detection-pipeline c3c81cb, which
# passed a one-pass Colab T4 Run all (2026-10-04); all 33 entries below carry that lock's versions and hashes unchanged.
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
cuda-pathfinder==1.8.2 \
    --hash=sha256:4e65059febdb4d19d5cbc4798677e19db2b582f2f702f457b609e571690d357e
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
filelock==4.0.4 \
    --hash=sha256:0df72be195ca7892216d16f2edce8d9b93a571f02402972020a8cff84c594c7b \
    --hash=sha256:90999ed63a26ccf86b93b959ab10cf1017f422d816be454ed54cbed263e71ab5
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
markupsafe==3.0.3 \
    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \
    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \
    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \
    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \
    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \
    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \
    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \
    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \
    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \
    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \
    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \
    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \
    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \
    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \
    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \
    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \
    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \
    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \
    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \
    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \
    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \
    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \
    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \
    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \
    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \
    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \
    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \
    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \
    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \
    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \
    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \
    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \
    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \
    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \
    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \
    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \
    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \
    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \
    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \
    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \
    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \
    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \
    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \
    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \
    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \
    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \
    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \
    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \
    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \
    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \
    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \
    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \
    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \
    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \
    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \
    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \
    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \
    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \
    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \
    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \
    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \
    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \
    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \
    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \
    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \
    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \
    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \
    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \
    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \
    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \
    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \
    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \
    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \
    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \
    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \
    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \
    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \
    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \
    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \
    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \
    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \
    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \
    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \
    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \
    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \
    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \
    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \
    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \
    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   maskrcnn-instance-segmentation-pipeline (pyproject.toml)
    #   torchvision
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   cuda-toolkit
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   maskrcnn-instance-segmentation-pipeline (pyproject.toml)
    #   torchvision
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   maskrcnn-instance-segmentation-pipeline (pyproject.toml)
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   maskrcnn-instance-segmentation-pipeline (pyproject.toml)
    #   torchvision
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   maskrcnn-instance-segmentation-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   -c <detr-detection-pipeline c3c81cb lock versions>
    #   torch
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

The next cell starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. These two cells are marked `# dimer: kernel cell` and are the only cells that run in the kernel. If you re-run a single cell later, it still runs in the isolated environment with the variables created so far; to start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Route the remaining cells to the isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import signal
from multiprocessing.connection import Connection

from IPython import get_ipython as _kernel_shell

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]


if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
else:
    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")

### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here: the locked environment already holds the pins listed in the cell. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `torchvision`, `numpy`, `PIL` versions, and whether CUDA is available. (The pip install and stale-import guard in the cell are skipped: the isolated worker, like an executor that has pre-installed exactly these pins, sets `DIMER_NOTEBOOK_CI_PREINSTALLED=1`.)

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'maskrcnn-instance-segmentation-pipeline',
    'repository_revision': '6244270d0ffbe6efee378fd1a73b542a0baa55c2',
    'embedded_module': 'src/maskrcnn_instance_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/maskrcnn_instance_segmentation_pipeline/pipeline.py', 'src/maskrcnn_instance_segmentation_pipeline/samples.py'],
    'module_sha256': '3579ab2decb6c691d43e32885bb37a80a54bd058ddd6d05dd04d0c0b7c4bdac7',
    'generator': 'build_notebook.py/2.1',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchvision, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/maskrcnn_instance_segmentation_pipeline/` @ `6244270d0ffb`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/maskrcnn_instance_segmentation_pipeline/pipeline.py`

In [ ]:
"""COCO instance segmentation and bounded fine-tuning with torchvision's Mask R-CNN ResNet-50 FPN v2 weights.

The model is torchvision's own ``maskrcnn_resnet50_fpn_v2`` architecture, built with ``weights=None`` so
that nothing is downloaded at construction, then loaded from one digest-verified local checkpoint file
(``weights/<key>/``) with ``torch.load(..., weights_only=True)``, which refuses arbitrary pickled objects.

Until ``tools/pin_snapshot.py`` has recorded the checkpoint's SHA-256 and byte size, the package refuses to
stage, verify or load weights: an unpinned checkpoint is never trusted.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "torchvision/maskrcnn_resnet50_fpn_v2"
# The pinned identity of a URL-hosted checkpoint is the SHA-256 of its bytes. torchvision names the file
# after the first 8 hex digits of that digest, so the URL is content-addressed; the full digest is pinned.
MODEL_REVISION = "73cbd0190fcbe3ba339921fbce2c3a0b6bb9126c9a133c85e43a2a8e060a109e"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "maskrcnn-resnet50-fpn-v2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "maskrcnn-adapter-v1"
UNPINNED = "unpinned"
PIN_COMMAND = "python tools/pin_snapshot.py"
WEIGHTS_FILE = "maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth"
WEIGHTS_URL = f"https://download.pytorch.org/models/{WEIGHTS_FILE}"
WEIGHTS_SHA256_PREFIX = "73cbd019"
TORCHVISION_WEIGHTS = "MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1"

# torchvision's COCO category list for these weights (MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1.meta
# ["categories"]), in label-id order: 91 slots. Id 0 is the background class and the 10 "N/A" ids are COCO
# category numbers that COCO 2017 never annotated, which leaves 80 object classes. LABELS drops the
# background, so model label id k is LABELS[k - 1].
COCO_CATEGORIES = (
    "__background__", "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck",
    "boat", "traffic light", "fire hydrant", "N/A", "stop sign", "parking meter", "bench", "bird",
    "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "N/A", "backpack",
    "umbrella", "N/A", "N/A", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard",
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket",
    "bottle", "N/A", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "N/A", "dining table", "N/A", "N/A", "toilet", "N/A", "tv", "laptop",
    "mouse", "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator",
    "N/A", "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush",
)  # fmt: skip
LABELS = COCO_CATEGORIES[1:]

# Score threshold: the value torchvision's visualization-utilities gallery uses to keep Mask R-CNN instances
# (score_threshold = .75, shown there with the v1 maskrcnn_resnet50_fpn weights), with its mask probability
# cut (proba_threshold = 0.5). Scores are softmax class probabilities after torchvision's per-class NMS;
# neither value was re-derived for the v2 weights or calibrated for any deployment, and the deployment owns
# tuning them on labelled images.
DETECTION_THRESHOLD = 0.75
MASK_THRESHOLD = 0.5
EVAL_DETECTION_THRESHOLD = 0.05
MAX_DETECTIONS = 100  # torchvision's box_detections_per_img default
MAX_EVAL_DETECTIONS = 100
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_CLASSES = 1000
MAX_RECORDS = 5000

# Training defaults for the bounded tutorial adaptation (torchvision's detection fine-tuning tutorial
# uses SGD, learning rate 0.005, momentum 0.9 and weight decay 0.0005).
DEFAULT_EPOCHS = 10
DEFAULT_BATCH_SIZE = 2
DEFAULT_LEARNING_RATE = 0.005
DEFAULT_MOMENTUM = 0.9
DEFAULT_WEIGHT_DECAY = 0.0005
DEFAULT_SEED = 20260924
BACKBONE_PREFIX = "backbone.body."

COCO_IOU_THRESHOLDS: tuple[float, ...] = tuple(round(0.50 + 0.05 * i, 2) for i in range(10))


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_pinned() -> bool:
    """True once MODEL_REVISION is the checkpoint's 64-hex SHA-256."""
    revision = MODEL_REVISION
    return len(revision) == 64 and all(c in "0123456789abcdef" for c in revision)


def _require_pinned(action: str) -> None:
    if not is_pinned():
        raise RuntimeError(
            f"refusing to {action}: {MODEL_ID} has no pinned checkpoint digest yet (MODEL_REVISION = "
            f"{MODEL_REVISION!r}); run `{PIN_COMMAND}` to record the SHA-256 and byte size"
        )


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local checkpoint against its DIMER manifest; raise naming the first mismatch."""
    _require_pinned("verify the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        if not entry.get("sha256") or entry.get("bytes") is None:
            raise ValueError(f"{entry['path']}: manifest records no sha256/bytes; run `{PIN_COMMAND}`")
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"checkpoint file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _url_download(entry: Mapping[str, Any], root: Path) -> None:
    """Fetch one manifest entry from its URL; torch.hub checks the filename's SHA-256 prefix on the way in."""
    from torch.hub import download_url_to_file

    download_url_to_file(
        entry["url"], str(root / entry["path"]), hash_prefix=WEIGHTS_SHA256_PREFIX, progress=False
    )


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[Mapping[str, Any], Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    _require_pinned("stage the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"checkpoint at {root} is missing {[e['path'] for e in missing]}; pass allow_download=True to "
            "fetch it"
        )
    fetch = downloader or _url_download
    for entry in missing:
        if entry.get("url") != WEIGHTS_URL:
            raise ValueError(f"manifest URL {entry.get('url')!r} != package URL {WEIGHTS_URL!r}")
        fetch(entry, root)
    return [entry["path"] for entry in missing]


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-union of two boolean masks of the same shape."""
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    if a.shape != b.shape:
        raise ValueError(f"mask shapes differ: {a.shape} vs {b.shape}")
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


def average_precision(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    kind: str = "box",
    iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
) -> dict[str, Any]:
    """Compact COCO-style average precision over a scored dataset, on boxes or on masks.

    For each class and IoU threshold, detections are matched greedily to references by descending score;
    each detection matches at most one reference, and precision is sampled at 101 recall points. Classes
    with no reference are left out of the mean. There are no area ranges and no crowd handling.
    """
    if kind not in ("box", "mask"):
        raise ValueError(f"kind must be 'box' or 'mask', got {kind!r}")
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} prediction lists but {len(references)} references")
    recall_points = np.linspace(0.0, 1.0, 101)
    per_threshold: dict[float, dict[str, float]] = {}

    def overlap(det: Mapping[str, Any], reference: Mapping[str, Any], j: int) -> float:
        if kind == "box":
            return box_iou(det["box"], reference["boxes"][j])
        return mask_iou(det["mask"], reference["masks"][j])

    for threshold in iou_thresholds:
        per_class: dict[str, float] = {}
        for name in class_names:
            scored: list[tuple[float, bool]] = []
            n_references = 0
            for dets, reference in zip(predictions, references, strict=True):
                ref_ids = [j for j, label in enumerate(reference["labels"]) if label == name]
                n_references += len(ref_ids)
                claimed = dict.fromkeys(ref_ids, False)
                for det in sorted((d for d in dets if d["label"] == name), key=lambda d: -float(d["score"])):
                    best, best_iou = -1, 0.0
                    for j in ref_ids:
                        if claimed[j]:
                            continue
                        value = overlap(det, reference, j)
                        if value > best_iou:
                            best, best_iou = j, value
                    hit = best >= 0 and best_iou >= threshold
                    if hit:
                        claimed[best] = True
                    scored.append((float(det["score"]), hit))
            if n_references == 0:
                continue
            if not scored:
                per_class[name] = 0.0
                continue
            scored.sort(key=lambda pair: -pair[0])
            true_positives = np.cumsum([1 if hit else 0 for _s, hit in scored])
            false_positives = np.cumsum([0 if hit else 1 for _s, hit in scored])
            recall = true_positives / n_references
            precision = np.maximum.accumulate(
                (true_positives / np.maximum(true_positives + false_positives, 1))[::-1]
            )[::-1]
            sampled = np.zeros_like(recall_points)
            indices = np.searchsorted(recall, recall_points, side="left")
            valid = indices < len(precision)
            sampled[valid] = precision[indices[valid]]
            per_class[name] = float(sampled.mean())
        per_threshold[threshold] = per_class
    means = {t: (float(np.mean(list(v.values()))) if v else 0.0) for t, v in per_threshold.items()}
    return {
        "ap": float(np.mean(list(means.values()))) if means else 0.0,
        "ap50": means.get(0.5, 0.0),
        "ap75": means.get(0.75, 0.0),
        "per_class_ap50": per_threshold.get(0.5, {}),
        "iou_thresholds": [float(t) for t in iou_thresholds],
        "kind": kind,
        "n_images": len(references),
        "n_references": sum(len(r["labels"]) for r in references),
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any, name: str = "threshold") -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


def _check_class_names(class_names: Sequence[str]) -> tuple[str, ...]:
    names = tuple(class_names)
    if not 1 <= len(names) <= MAX_CLASSES:
        raise ValueError(f"class_names must hold 1..{MAX_CLASSES} names, got {len(names)}")
    for name in names:
        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"class names must be non-empty strings, got {name!r}")
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"class_names contains duplicates: {duplicates}")
    return names


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB): a photograph or a rendered scene",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "mask_threshold": MASK_THRESHOLD,
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB and to a float tensor in [0, 1]; the model's own transform resizes it so the "
        "shorter side is 800 px and the longer at most 1333 px and normalises it with the ImageNet mean and "
        "standard deviation; boxes and masks are returned in input pixels"
    ),
}

DATASET_SCHEMA: dict[str, Any] = {
    "record": (
        "{'image': PIL.Image.Image, 'boxes': [[x0, y0, x1, y1], ...], 'labels': [name, ...], "
        "'masks': [bool array (height, width), ...]}"
    ),
    "boxes": "xyxy pixel coordinates inside the image, x0 < x1 and y0 < y1, one per instance",
    "masks": "one non-empty boolean mask per instance, the size of the image, lying inside its box",
    "labels": "one name per instance, each one of class_names",
    "records": [1, MAX_RECORDS],
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request."""
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict)."""
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    epochs: int = DEFAULT_EPOCHS,
) -> dict[str, Any]:
    """Validation stage for labelled instance-segmentation records: raise on the first broken record, else
    return the dataset manifest. Classes that never occur are reported as findings, not silently accepted."""
    names = _check_class_names(class_names)
    if not records:
        raise ValueError("dataset must hold at least one record")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset holds {len(records)} records > MAX_RECORDS {MAX_RECORDS}")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
        raise ValueError(f"epochs must be an int in 1..100, got {epochs!r}")
    valid = set(names)
    per_class = dict.fromkeys(names, 0)

    def _pixel_hint(box_values: Sequence[float]) -> str:
        # MRC-m1: boxes in [0, 1] are a common BYOD mistake; say so, not only which rule failed.
        if all(0.0 <= v <= 1.0 for v in box_values):
            return (
                "; every coordinate of this box is between 0 and 1, which looks like normalised coordinates: "
                "boxes must be xyxy pixel coordinates (multiply x by the image width and y by its height)"
            )
        return ""

    total = 0
    for idx, record in enumerate(records):
        if not isinstance(record, Mapping) or not {"image", "boxes", "labels", "masks"} <= set(record):
            raise ValueError(f"record {idx} must be a mapping with 'image', 'boxes', 'labels' and 'masks'")
        try:
            img = validate_image(record["image"])
        except (TypeError, ValueError) as exc:
            raise type(exc)(f"record {idx}: {exc}") from exc
        boxes, labels, masks = record["boxes"], record["labels"], record["masks"]
        if not len(boxes) == len(labels) == len(masks):
            raise ValueError(f"record {idx}: {len(boxes)} boxes, {len(labels)} labels and {len(masks)} masks")
        width, height = img.size
        for b_idx, (box, mask) in enumerate(zip(boxes, masks, strict=True)):
            if len(box) != 4:
                raise ValueError(f"record {idx} box {b_idx} must have 4 elements, got {len(box)}")
            x0, y0, x1, y1 = (float(v) for v in box)
            if not all(np.isfinite([x0, y0, x1, y1])):
                raise ValueError(f"record {idx} box {b_idx} has a non-finite coordinate")
            if not (0.0 <= x0 < x1 <= width and 0.0 <= y0 < y1 <= height):
                raise ValueError(
                    f"record {idx} box {b_idx} [{x0}, {y0}, {x1}, {y1}] is empty or outside image bounds "
                    f"{(width, height)}" + _pixel_hint((x0, y0, x1, y1))
                )
            array = np.asarray(mask)
            if array.shape != (height, width):
                raise ValueError(
                    f"record {idx} mask {b_idx} has shape {array.shape}, image is {(height, width)}"
                )
            if array.dtype != bool and not np.isin(array, (0, 1)).all():
                raise ValueError(f"record {idx} mask {b_idx} must be boolean or 0/1")
            ys, xs = np.nonzero(array)
            if not len(xs):
                raise ValueError(f"record {idx} mask {b_idx} is empty")
            if xs.min() < x0 - 1 or xs.max() + 1 > x1 + 1 or ys.min() < y0 - 1 or ys.max() + 1 > y1 + 1:
                raise ValueError(
                    f"record {idx} mask {b_idx} extends outside its box" + _pixel_hint((x0, y0, x1, y1))
                )
        for label in labels:
            if label not in valid:
                raise ValueError(f"record {idx} has unknown class {label!r}; expected one of {list(names)}")
            per_class[label] += 1
        total += len(labels)
    return {
        "schema": dict(DATASET_SCHEMA),
        "n_records": len(records),
        "n_instances": total,
        "class_names": list(names),
        "instances_per_class": per_class,
        "epochs": epochs,
        "findings": [
            f"class {name!r} has no instance in this dataset" for name, n in per_class.items() if n == 0
        ],
        "verdict": "accepted",
    }


def _open_named(idx: int, kind: str, path: Path, root: Path) -> Image.Image:
    """Open one BYOD file fully, or raise a ValueError that names the file and the fix (MRC-m1)."""
    relative = path.relative_to(root) if root in path.parents else path
    if not path.is_file():
        raise ValueError(
            f"annotations.json entry {idx}: {kind} file {str(relative)!r} is missing from {root}; "
            "add the file or correct its name in annotations.json"
        )
    try:
        with Image.open(path) as handle:
            handle.load()
            return handle.copy()
    except (OSError, SyntaxError, ValueError) as exc:  # PIL.UnidentifiedImageError is an OSError
        raise ValueError(
            f"annotations.json entry {idx}: {kind} file {str(relative)!r} is not an image Pillow can "
            f"read ({type(exc).__name__}); save it as PNG (masks: single-channel PNG, non-zero inside "
            "the instance)"
        ) from exc


def read_detection_records(directory: str | Path) -> list[dict[str, Any]]:
    """Read BYOD records from ``<directory>/annotations.json`` plus the image and mask files it names.

    ``annotations.json`` is a list of ``{"file": "name.png", "boxes": [[x0, y0, x1, y1], ...],
    "labels": [...], "masks": ["name_mask0.png", ...]}`` objects; each mask file is a single-channel PNG
    whose non-zero pixels are the instance. File names must stay inside ``directory``; absolute paths and
    ``..`` are refused before any file is opened. The result still has to pass ``validate_dataset``.
    """
    root = Path(directory).resolve()
    index_path = root / "annotations.json"
    if not index_path.is_file():
        raise FileNotFoundError(f"{index_path} not found; expected annotations.json next to the images")
    with open(index_path, encoding="utf-8") as fh:
        entries = json.load(fh)
    if not isinstance(entries, list) or not entries:
        raise ValueError("annotations.json must hold a non-empty list of records")
    if len(entries) > MAX_RECORDS:
        raise ValueError(f"annotations.json lists {len(entries)} records > MAX_RECORDS {MAX_RECORDS}")

    def resolve(idx: int, name: Any) -> Path:
        relative = Path(str(name))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"annotations.json entry {idx}: file {name!r} must be relative to {root}")
        resolved = (root / relative).resolve()
        if root not in resolved.parents:
            raise ValueError(f"annotations.json entry {idx}: file {name!r} resolves outside {root}")
        return resolved

    records: list[dict[str, Any]] = []
    for idx, entry in enumerate(entries):
        if not isinstance(entry, dict) or not {"file", "boxes", "labels", "masks"} <= set(entry):
            raise ValueError(f"annotations.json entry {idx} must have 'file', 'boxes', 'labels' and 'masks'")
        image_path = resolve(idx, entry["file"])
        mask_paths = [resolve(idx, name) for name in entry["masks"]]
        image = _open_named(idx, "image", image_path, root).convert("RGB")
        masks = [np.asarray(_open_named(idx, "mask", path, root).convert("L")) > 0 for path in mask_paths]
        records.append(
            {
                "id": str(entry["file"]),
                "image": image,
                "boxes": entry["boxes"],
                "labels": entry["labels"],
                "masks": masks,
            }
        )
    return records


def evaluation_report(
    result: Mapping[str, Any],
    references: Mapping[str, Mapping[str, Sequence[Any]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Single-image evaluation stage: per-object box_iou and mask_iou against same-label detections."""
    detections = list(result["detections"])
    masks = list(result.get("masks") or [])
    labels = tuple(result.get("class_names") or LABELS)
    base = {
        "task": f"instance segmentation over {len(labels)} class slots on one image",
        "decision_rule": (
            "an instance survives when its softmax class probability reaches the threshold after "
            "torchvision's "
            "per-class non-maximum suppression; its mask is the pixels whose probability is at least "
            f"{MASK_THRESHOLD}; "
            "neither value is a calibrated probability for the deployment's images"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "mask_threshold": MASK_THRESHOLD,
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not references:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference boxes or masks were supplied for the evaluated image",
            "needs": (
                "labelled instance masks per class on your own images, scored per object with box_iou and "
                "mask_iou "
                "and aggregated into box and mask average precision at stated IoU thresholds"
            ),
        }
    metrics = []
    for label, reference in references.items():
        if label not in labels:
            raise ValueError(f"unknown reference label {label!r}; expected one of {len(labels)} classes")
        same = [i for i, det in enumerate(detections) if det["label"] == label]
        for index, box in enumerate(reference["boxes"]):
            box_ious = [box_iou(detections[i]["box"], box) for i in same]
            best = same[int(np.argmax(box_ious))] if box_ious else None
            ref_mask = reference.get("masks", [None] * len(reference["boxes"]))[index]
            metrics.append(
                {
                    "id": "box_iou+mask_iou",
                    "reference": f"{label}-{index}",
                    "box_iou": max(box_ious) if box_ious else 0.0,
                    "mask_iou": mask_iou(masks[best], ref_mask)
                    if best is not None and ref_mask is not None and masks
                    else 0.0,
                    "matched_score": detections[best]["score"] if best is not None else None,
                    "n_detected_same_label": len(same),
                    "estimation": "one reference instance per object on a single image, no dispersion "
                    "estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": f"{len(metrics)} reference instance(s) on one tutorial image; geometry sanity evidence, "
        "not a benchmark",
        "needs": "a labelled image set from the deployment domain for any average-precision or "
        "precision/recall claim",
    }


def build_model(num_classes: int = len(COCO_CATEGORIES)) -> Any:
    """torchvision's maskrcnn_resnet50_fpn_v2 architecture with random weights and nothing downloaded."""
    from torchvision.models.detection import maskrcnn_resnet50_fpn_v2

    return maskrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=num_classes)


def _rehead(model: Any, num_labels: int) -> tuple[str, ...]:
    """Replace the box and mask predictors for ``num_labels`` classes plus background."""
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

    box_in = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(box_in, num_labels + 1)
    mask_in = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_in, 256, num_labels + 1)
    return ("roi_heads.box_predictor.", "roi_heads.mask_predictor.")


def _hold_batchnorm(module: Any) -> None:
    """Keep every BatchNorm layer under ``module`` in eval mode, so frozen layers keep their statistics."""
    import torch

    for sub in module.modules():
        if isinstance(sub, torch.nn.modules.batchnorm._BatchNorm):
            sub.eval()


@dataclass
class MaskRcnnPipeline:
    """COCO-class instance segmentation and transfer fine-tuning over Mask R-CNN ResNet-50 FPN v2."""

    model: Any
    device: str
    class_names: tuple[str, ...] = LABELS
    source: str = "snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    reinitialised: tuple[str, ...] = field(default_factory=tuple)
    frozen_prefixes: tuple[str, ...] = field(default_factory=tuple)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        class_names: Sequence[str] | None = None,
        seed: int = DEFAULT_SEED,
    ) -> MaskRcnnPipeline:
        _require_pinned("load the model")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no checkpoint manifest at {root}; stage {MODEL_ID} under weights/{MODEL_KEY} "
                "(allow_download=True fetches the manifest-listed file)"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        names = _check_class_names(class_names) if class_names is not None else LABELS

        import torch

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = build_model(len(COCO_CATEGORIES))
        state = torch.load(root / WEIGHTS_FILE, map_location="cpu", weights_only=True)
        model.load_state_dict(state, strict=True)
        reinitialised: tuple[str, ...] = ()
        if class_names is not None:
            torch.manual_seed(seed)
            reinitialised = _rehead(model, len(names))
        model = model.to(resolved_device).eval()
        return cls(
            model=model,
            device=resolved_device,
            class_names=names,
            source=str(root),
            base_state_digest=MODEL_REVISION,
            reinitialised=reinitialised,
        )

    def _run(self, image: Image.Image, threshold: float) -> tuple[list[dict[str, Any]], list[np.ndarray]]:
        import torch
        from torchvision.transforms.functional import to_tensor

        was_training = self.model.training
        self.model.eval()
        with torch.inference_mode():
            output = self.model([to_tensor(image).to(self.device)])[0]
        if was_training:
            self.model.train()
        detections, masks = [], []
        for box, label, score, mask in zip(
            output["boxes"], output["labels"], output["scores"], output["masks"], strict=True
        ):
            if float(score) < threshold:
                continue
            idx = int(label) - 1
            name = self.class_names[idx] if 0 <= idx < len(self.class_names) else f"class_{int(label)}"
            binary = (mask[0] >= MASK_THRESHOLD).cpu().numpy()
            detections.append(
                {
                    "box": [float(v) for v in box.tolist()],
                    "label": name,
                    "score": float(score),
                    "mask_area": int(binary.sum()),
                }
            )
            masks.append(binary)
        return detections, masks

    def detect(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Segment instances on one image; boxes are xyxy input pixels, masks boolean (height, width)."""
        rgb, checked = _check_inputs(image, threshold)
        detections, masks = self._run(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > MAX_DETECTIONS {MAX_DETECTIONS}"
            )
        for det, mask in zip(detections, masks, strict=True):
            if (
                set(det) != {"box", "label", "score", "mask_area"}
                or len(det["box"]) != 4
                or det["label"] not in self.class_names
            ):
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
            if mask.shape != (rgb.height, rgb.width):
                raise RuntimeError(f"backend returned a mask of shape {mask.shape} for a {rgb.size} image")
        order = sorted(range(len(detections)), key=lambda i: -detections[i]["score"])
        return {
            "detections": [detections[i] for i in order],
            "masks": [masks[i] for i in order],
            "threshold": checked,
            "mask_threshold": MASK_THRESHOLD,
            "width": rgb.width,
            "height": rgb.height,
            "class_names": list(self.class_names),
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        seed: int = DEFAULT_SEED,
        freeze_backbone: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning on ``records`` with torchvision's own Mask R-CNN losses.

        The loss is the sum torchvision returns in training mode: RPN objectness and box regression, ROI
        classification and box regression, and the per-pixel mask cross-entropy. With ``freeze_backbone`` the
        ResNet-50 body keeps its weights and its BatchNorm statistics (its BatchNorm layers are held in eval
        mode); the FPN, the RPN and the ROI heads are trained. Mutates this pipeline in place.

        A pipeline that is already adapted (fine-tuned, or loaded with an adapter) is refused: training it
        again would stack a second schedule on the first while the returned record describes only the
        second. Build a fresh re-headed pipeline with ``from_pretrained(class_names=..., seed=...)`` instead.
        """
        if self.adapted:
            raise RuntimeError(
                "Cannot fine-tune: this pipeline was already adapted (fine-tuned, or loaded with an "
                "adapter), so a second call would continue from the adapted weights while the run record "
                "described only the new schedule. Rebuild it with MaskRcnnPipeline.from_pretrained("
                "weights_dir=..., class_names=..., seed=...) and fine-tune that (the tutorial's "
                "reset_to_pretrained() does this)."
            )
        import torch
        from torchvision.transforms.functional import to_tensor

        validate_dataset(records, self.class_names, epochs=epochs)
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if (
            isinstance(learning_rate, bool)
            or not isinstance(learning_rate, int | float)
            or not 0.0 < float(learning_rate) <= 1.0
        ):
            raise ValueError(f"learning_rate must be a number in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)
        for name, parameter in self.model.named_parameters():
            parameter.requires_grad = not (freeze_backbone and name.startswith(BACKBONE_PREFIX))
        self.frozen_prefixes = (BACKBONE_PREFIX,) if freeze_backbone else ()
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.SGD(
            trainable, lr=float(learning_rate), momentum=DEFAULT_MOMENTUM, weight_decay=DEFAULT_WEIGHT_DECAY
        )
        class_to_id = {name: i + 1 for i, name in enumerate(self.class_names)}

        epoch_losses: list[float] = []
        for epoch in range(epochs):
            self.model.train()
            if freeze_backbone:
                _hold_batchnorm(self.model.backbone.body)
            order = rng.permutation(len(records))
            running, n_batches = 0.0, 0
            for start in range(0, len(records), batch_size):
                batch = [records[int(i)] for i in order[start : start + batch_size]]
                images = [to_tensor(validate_image(r["image"])).to(self.device) for r in batch]
                targets = [
                    {
                        "boxes": torch.tensor(
                            [[float(v) for v in b] for b in r["boxes"]],
                            dtype=torch.float32,
                            device=self.device,
                        ),
                        "labels": torch.tensor(
                            [class_to_id[n] for n in r["labels"]], dtype=torch.int64, device=self.device
                        ),
                        "masks": torch.as_tensor(
                            np.stack([np.asarray(m, dtype=np.uint8) for m in r["masks"]]), device=self.device
                        ),
                    }
                    for r in batch
                ]
                optimizer.zero_grad(set_to_none=True)
                losses = self.model(images, targets)
                loss = sum(losses.values())
                loss.backward()
                optimizer.step()
                running += float(loss.detach().cpu())
                n_batches += 1
            epoch_losses.append(running / max(1, n_batches))
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": epoch_losses[-1]})
        self.model.eval()
        self.adapted = True
        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "momentum": DEFAULT_MOMENTUM,
            "weight_decay": DEFAULT_WEIGHT_DECAY,
            "optimizer": "SGD",
            "seed": seed,
            "precision": "float32",
            "freeze_backbone": freeze_backbone,
            "frozen_prefixes": list(self.frozen_prefixes),
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "loss": "torchvision Mask R-CNN training losses (RPN objectness + RPN box + ROI class + ROI "
            "box + mask)",
            "device": self.device,
            "class_names": list(self.class_names),
            "reinitialised_prefixes": list(self.reinitialised),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
        iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
        max_detections: int = MAX_EVAL_DETECTIONS,
    ) -> dict[str, Any]:
        """Score a labelled dataset: box and mask average precision plus evaluation metadata."""
        checked = _check_threshold(threshold, "threshold")
        predictions = []
        for record in records:
            result = self.detect(record["image"], threshold=checked)
            dets = [{**d, "mask": m} for d, m in zip(result["detections"], result["masks"], strict=True)][
                :max_detections
            ]
            predictions.append(dets)
        box = average_precision(
            predictions, records, self.class_names, kind="box", iou_thresholds=iou_thresholds
        )
        mask = average_precision(
            predictions, records, self.class_names, kind="mask", iou_thresholds=iou_thresholds
        )
        return {
            "ap": box["ap"],
            "ap50": box["ap50"],
            "ap75": box["ap75"],
            "per_class_ap50": box["per_class_ap50"],
            "mask_ap": mask["ap"],
            "mask_ap50": mask["ap50"],
            "mask_ap75": mask["ap75"],
            "per_class_mask_ap50": mask["per_class_ap50"],
            "n_images": box["n_images"],
            "n_references": box["n_references"],
            "threshold": checked,
            "mask_threshold": MASK_THRESHOLD,
            "max_detections": max_detections,
            "adapted": self.adapted,
            "class_names": list(self.class_names),
            "estimation": f"one pass over {len(records)} held-out images; no resampling, no dispersion "
            "estimate",
            "implementation": (
                "package-local average_precision on boxes and on masks: greedy score-ordered matching and "
                "101-point "
                "interpolation, without pycocotools area ranges or crowd handling"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write the adapted tensors as one SafeTensors file with the provenance in its metadata.

        Tensors under ``frozen_prefixes`` are left out: they equal the verified base checkpoint, which
        ``load_artifact`` loads first. The artifact is therefore an adapter bound to the base checkpoint.
        """
        from safetensors.torch import save_file

        if not self.adapted:
            raise RuntimeError("nothing to export: the pipeline has not been fine-tuned")
        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        tensors = {
            name: value.detach().cpu().contiguous().clone()
            for name, value in self.model.state_dict().items()
            if not any(name.startswith(prefix) for prefix in self.frozen_prefixes)
        }
        metadata = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "class_names": json.dumps(list(self.class_names)),
            "frozen_prefixes": json.dumps(list(self.frozen_prefixes)),
            "base_state_digest": self.base_state_digest or "",
            "notes": notes or "",
        }
        save_file(tensors, str(artifact_path), metadata=metadata)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "class_names": list(self.class_names),
            "tensors": len(tensors),
            "frozen_prefixes": list(self.frozen_prefixes),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @staticmethod
    def read_artifact_metadata(path: str | Path) -> dict[str, Any]:
        """Read and check the artifact's provenance header without loading any tensor."""
        from safetensors import safe_open

        with safe_open(str(path), framework="pt") as handle:
            metadata = dict(handle.metadata() or {})
        if metadata.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {metadata.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if (metadata.get("model_id"), metadata.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(
                f"artifact was built on {metadata.get('model_id')}@{metadata.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if metadata.get("model_key") != MODEL_KEY:
            raise ValueError(
                f"artifact was built on {metadata.get('model_key')!r}, package pins {MODEL_KEY!r}"
            )
        return {
            **metadata,
            "class_names": json.loads(metadata["class_names"]),
            "frozen_prefixes": json.loads(metadata["frozen_prefixes"]),
        }

    def apply_artifact(self, path: str | Path) -> None:
        """Load adapter tensors onto this (base) pipeline; refuse any tensor set that does not fit."""
        from safetensors.torch import load_file

        metadata = self.read_artifact_metadata(path)
        if tuple(metadata["class_names"]) != tuple(self.class_names):
            raise ValueError("artifact class_names differ from this pipeline's class_names")
        expected_base = metadata.get("base_state_digest") or None
        if expected_base and self.base_state_digest and expected_base != self.base_state_digest:
            raise ValueError("artifact was exported against a different base checkpoint digest")
        tensors = load_file(str(path), device="cpu")
        prefixes = tuple(metadata["frozen_prefixes"])
        result = self.model.load_state_dict(tensors, strict=False)
        if result.unexpected_keys:
            raise ValueError(
                f"artifact carries tensors the model does not have: {result.unexpected_keys[:5]}"
            )
        stray = [key for key in result.missing_keys if not any(key.startswith(p) for p in prefixes)]
        if stray:
            raise ValueError(f"artifact is missing trainable tensors: {stray[:5]}")
        self.model.eval()
        self.adapted = True
        self.frozen_prefixes = prefixes
        self.source = f"artifact:{Path(path).name}"

    @classmethod
    def load_artifact(
        cls,
        path: str | Path,
        *,
        weights_dir: str | Path | None = None,
        device: str | None = None,
    ) -> MaskRcnnPipeline:
        """Rebuild an adapted pipeline: verified base checkpoint first, then the adapter tensors."""
        metadata = cls.read_artifact_metadata(path)
        pipe = cls.from_pretrained(
            device=device, weights_dir=weights_dir, class_names=metadata["class_names"]
        )
        pipe.apply_artifact(path)
        return pipe

**Module 2/2:** `src/maskrcnn_instance_segmentation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data with exact instance masks: the COCO demonstration scene and the
adaptation dataset.

Nothing here is downloaded and nothing needs torch — Pillow and numpy only. Every object is drawn twice:
once onto the RGB image and once onto its own single-channel mask, with the same geometry, so each
reference mask is exact by construction and each reference box is the tight box of that mask.

Two separate label vocabularies live here and must not be conflated:

* ``tutorial_scene`` returns references drawn from the checkpoint's own **COCO** classes; it
  demonstrates the pretrained model and is not training data.
* ``sign_dataset`` returns records labelled with ``SIGN_CLASSES``, a three-class vocabulary that does
  **not** exist in COCO. It is the adaptation dataset, and a model fine-tuned on it answers in those
  three names only.
"""

from __future__ import annotations

import math
from collections.abc import Callable
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageFont

COCO_SCENE_SIZE = (640, 480)
ADAPT_SCENE_SIZE = (640, 640)

# The adaptation vocabulary. Deliberately not COCO names: "stop sign" exists in COCO, "yield-sign" and
# "speed-limit-sign" do not, and the hyphenated spellings keep the two vocabularies visually distinct.
SIGN_CLASSES: tuple[str, ...] = ("stop-sign", "yield-sign", "speed-limit-sign")


def _font(size: int) -> Any:
    return ImageFont.load_default(size=size)


def _mask_and_box(
    size: tuple[int, int], shape: Callable[[ImageDraw.ImageDraw], None]
) -> tuple[np.ndarray, list[float]]:
    """Draw ``shape`` filled onto an empty mask; return the boolean mask and its tight xyxy box."""
    mask_image = Image.new("L", size, 0)
    shape(ImageDraw.Draw(mask_image))
    mask = np.asarray(mask_image) > 0
    ys, xs = np.nonzero(mask)
    if not len(xs):
        raise ValueError("shape drew nothing")
    return mask, [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def tutorial_scene(
    width: int = COCO_SCENE_SIZE[0], height: int = COCO_SCENE_SIZE[1]
) -> tuple[Image.Image, dict[str, dict[str, Any]]]:
    """The COCO demonstration scene: drawn objects with reference boxes and masks, keyed by COCO label.

    Returns ``(image, {label: {"boxes": [[x0, y0, x1, y1]], "masks": [bool HxW array]}})``. These are drawn
    references on a rendered picture, not a labelled photographic dataset: they support a per-object
    ``box_iou`` / ``mask_iou`` sanity check and nothing more.
    """
    size = (width, height)
    img = Image.new("RGB", size, (135, 190, 235))
    d = ImageDraw.Draw(img)
    d.rectangle([0, int(height * 0.6875), width, height], fill=(96, 128, 72))
    d.rectangle([0, int(height * 0.625), width, int(height * 0.6875)], fill=(110, 110, 110))
    refs: dict[str, dict[str, Any]] = {}

    def add(label: str, shape: Callable[[ImageDraw.ImageDraw], None]) -> None:
        mask, box = _mask_and_box(size, shape)
        refs[label] = {"boxes": [box], "masks": [mask]}

    # Stop sign (the octagon only; the post is scenery)
    cx, cy, r = 110, 150, 62
    pts = [
        (cx + r * math.cos(math.pi / 8 + k * math.pi / 4), cy + r * math.sin(math.pi / 8 + k * math.pi / 4))
        for k in range(8)
    ]
    d.rectangle([cx - 5, cy, cx + 5, int(height * 0.6875)], fill=(90, 90, 90))
    d.polygon(pts, fill=(200, 20, 30), outline=(255, 255, 255))
    f = _font(30)
    d.text((cx - d.textlength("STOP", font=f) / 2, cy - 17), "STOP", fill="white", font=f)
    add("stop sign", lambda m, pts=pts: m.polygon(pts, fill=255, outline=255))

    # Traffic light (the housing)
    x0, y0 = 270, 60
    d.rectangle([x0 + 22, y0 + 150, x0 + 30, int(height * 0.6875)], fill=(70, 70, 70))
    d.rectangle([x0, y0, x0 + 52, y0 + 150], fill=(25, 25, 25), outline=(60, 60, 60))
    for k, col in enumerate([(230, 30, 30), (240, 200, 30), (40, 200, 60)]):
        d.ellipse([x0 + 8, y0 + 8 + k * 47, x0 + 44, y0 + 44 + k * 47], fill=col)
    add("traffic light", lambda m, x0=x0, y0=y0: m.rectangle([x0, y0, x0 + 52, y0 + 150], fill=255))

    # Clock (the face)
    cx, cy, r = 480, 140, 70
    d.rectangle([cx - 6, cy, cx + 6, int(height * 0.6875)], fill=(120, 80, 40))
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 250, 245), outline=(20, 20, 20), width=5)
    f2 = _font(16)
    for h in range(1, 13):
        a = math.radians(h * 30 - 90)
        d.text(
            (cx + (r - 18) * math.cos(a) - 5, cy + (r - 18) * math.sin(a) - 8), str(h), fill="black", font=f2
        )
    d.line(
        [(cx, cy), (cx + 0.5 * r * math.cos(math.radians(-60)), cy + 0.5 * r * math.sin(math.radians(-60)))],
        fill="black",
        width=5,
    )
    d.line(
        [(cx, cy), (cx + 0.8 * r * math.cos(math.radians(30)), cy + 0.8 * r * math.sin(math.radians(30)))],
        fill="black",
        width=3,
    )
    add("clock", lambda m, cx=cx, cy=cy, r=r: m.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255))

    # Sports ball
    cx, cy, r = 330, 400, 45
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(235, 120, 30), outline=(40, 20, 10), width=3)
    d.line([(cx - r, cy), (cx + r, cy)], fill=(40, 20, 10), width=3)
    d.line([(cx, cy - r), (cx, cy + r)], fill=(40, 20, 10), width=3)
    d.arc([cx - r * 1.6, cy - r, cx - r * 0.2, cy + r], 300, 60, fill=(40, 20, 10), width=3)
    d.arc([cx + r * 0.2, cy - r, cx + r * 1.6, cy + r], 120, 240, fill=(40, 20, 10), width=3)
    add("sports ball", lambda m, cx=cx, cy=cy, r=r: m.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255))

    return img, refs


def blank_scene(width: int = 640, height: int = 640) -> Image.Image:
    """A featureless white image: the degenerate input every detector should be asked about."""
    return Image.new("RGB", (width, height), (255, 255, 255))


def noise_scene(seed: int = 0, width: int = 640, height: int = 640) -> Image.Image:
    """Uniform RGB noise: structure-free input, for the same reason as ``blank_scene``."""
    rng = np.random.default_rng(seed)
    return Image.fromarray(rng.integers(0, 256, (height, width, 3), dtype=np.uint8))


def _octagon_points(cx: float, cy: float, r: float) -> list[tuple[float, float]]:
    return [
        (cx + r * math.cos(math.pi / 8 + i * math.pi / 4), cy + r * math.sin(math.pi / 8 + i * math.pi / 4))
        for i in range(8)
    ]


def _draw_sign(
    draw: ImageDraw.ImageDraw, kind: str, cx: float, cy: float, r: float, limit: int
) -> Callable[[ImageDraw.ImageDraw], None]:
    """Draw one sign onto the RGB canvas; return the function that draws its filled silhouette on a mask."""
    if kind == "stop-sign":
        points = _octagon_points(cx, cy, r)
        draw.polygon(points, fill=(196, 30, 34), outline=(255, 255, 255))
        font = _font(int(max(11, r * 0.34)))
        draw.text(
            (cx - draw.textlength("STOP", font=font) / 2, cy - r * 0.27),
            "STOP",
            fill=(255, 255, 255),
            font=font,
        )
        return lambda m: m.polygon(points, fill=255, outline=255)
    if kind == "yield-sign":
        points = [(cx, cy + r), (cx - r * 0.95, cy - r * 0.75), (cx + r * 0.95, cy - r * 0.75)]
        draw.polygon(points, fill=(255, 255, 255), outline=(198, 32, 36))
        inner = [(cx, cy + r * 0.62), (cx - r * 0.62, cy - r * 0.5), (cx + r * 0.62, cy - r * 0.5)]
        draw.line([*inner, inner[0]], fill=(198, 32, 36), width=int(max(4, r * 0.22)))
        return lambda m: m.polygon(points, fill=255, outline=255)
    box = [cx - r, cy - r, cx + r, cy + r]
    draw.ellipse(box, fill=(255, 255, 255), outline=(198, 32, 36), width=int(max(4, r * 0.2)))
    font = _font(int(max(12, r * 0.9)))
    text = str(limit)
    draw.text((cx - draw.textlength(text, font=font) / 2, cy - r * 0.55), text, fill=(30, 30, 30), font=font)
    return lambda m: m.ellipse(box, fill=255)


def sign_dataset(
    n_images: int = 40,
    *,
    seed: int = 0,
    max_objects: int = 3,
    size: tuple[int, int] = ADAPT_SCENE_SIZE,
) -> list[dict[str, Any]]:
    """A deterministic labelled instance-segmentation dataset over ``SIGN_CLASSES`` for the bounded fine-tune.

    Each record is ``{"id": str, "image": PIL.Image, "boxes": [[x0, y0, x1, y1], ...], "labels": [name, ...],
    "masks": [bool HxW array, ...]}`` — the record shape ``validate_dataset`` and ``finetune`` accept, and
    the shape a BYOD caller must produce. Signs sit on a non-overlapping grid, so every mask is the sign's
    whole silhouette and every box is the tight box of its mask.

    It is synthetic drawn data, so a model fine-tuned on it learns to segment *these renderings*. That is
    the point of a bounded tutorial adaptation and the reason its metrics are not a claim about real
    traffic signs.
    """
    if not 1 <= n_images <= 500:
        raise ValueError(f"n_images must be in 1..500, got {n_images}")
    if not 1 <= max_objects <= 6:
        raise ValueError(f"max_objects must be in 1..6, got {max_objects}")
    rng = np.random.default_rng(seed)
    width, height = size
    slots = [(x, y) for y in (150, 400) for x in (140, 360, 560)]
    records: list[dict[str, Any]] = []
    for index in range(n_images):
        image = Image.new("RGB", size, (232, 236, 240))
        draw = ImageDraw.Draw(image)
        tint = rng.integers(200, 245, 3)
        draw.rectangle([0, 0, width, height], fill=tuple(int(v) for v in tint))
        draw.rectangle([0, int(height * 0.72), width, height], fill=(118, 122, 126))
        n_objects = int(rng.integers(1, max_objects + 1))
        chosen = rng.permutation(len(slots))[:n_objects]
        boxes: list[list[float]] = []
        labels: list[str] = []
        masks: list[np.ndarray] = []
        for slot_index in chosen:
            cx, cy = slots[int(slot_index)]
            cx += float(rng.integers(-28, 29))
            cy += float(rng.integers(-28, 29))
            radius = float(rng.integers(46, 71))
            cx = min(max(cx, radius + 6), width - radius - 6)
            cy = min(max(cy, radius + 6), height - radius - 100)
            kind = SIGN_CLASSES[int(rng.integers(0, len(SIGN_CLASSES)))]
            limit = int(rng.choice([30, 50, 60, 80]))
            draw.rectangle(
                [cx - 5, cy + radius * 0.6, cx + 5, min(height - 1, cy + radius + 90)], fill=(112, 112, 116)
            )
            silhouette = _draw_sign(draw, kind, cx, cy, radius, limit)
            mask, box = _mask_and_box(size, silhouette)
            boxes.append(box)
            labels.append(kind)
            masks.append(mask)
        records.append(
            {
                "id": f"sign-{seed}-{index:03d}",
                "image": image,
                "boxes": boxes,
                "labels": labels,
                "masks": masks,
            }
        )
    return records


def split_dataset(
    records: list[dict[str, Any]],
    *,
    train_fraction: float = 0.75,
    seed: int = 0,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Deterministically partition records into train and held-out splits (random, record-level).

    A random split assumes the records are independent. Records that share a source photograph, a
    scene or a camera session must be split by that group instead, or the held-out score leaks.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
    if len(records) < 2:
        raise ValueError(f"at least 2 records are required to split, got {len(records)}")
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))
    n_train = min(len(records) - 1, max(1, int(len(records) * train_fraction)))
    train_idx = {int(i) for i in indices[:n_train]}
    train = [r for i, r in enumerate(records) if i in train_idx]
    held_out = [r for i, r in enumerate(records) if i not in train_idx]
    return train, held_out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the manifest URL on `download.pytorch.org` **and must hash to SHA-256 `73cbd0190fcb…`**, `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_url_snapshot",
  "formatVersion": 1,
  "modelKey": "maskrcnn-resnet50-fpn-v2",
  "modelId": "torchvision/maskrcnn_resnet50_fpn_v2",
  "revision": "73cbd0190fcbe3ba339921fbce2c3a0b6bb9126c9a133c85e43a2a8e060a109e",
  "files": [
    {
      "path": "maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth",
      "url": "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth",
      "bytes": 185828065,
      "sha256": "73cbd0190fcbe3ba339921fbce2c3a0b6bb9126c9a133c85e43a2a8e060a109e"
    }
  ],
  "totalBytes": 185828065
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the pretrained model does on a drawn scene

Before adapting anything, inspect the model you start from. The carried `samples` module draws a deterministic street scene with four objects, each with a reference box and an exact reference mask: a **stop sign**, a **traffic light**, an analogue **clock**, and an orange **sports ball**. All four are COCO classes.

The detection threshold is a **caller-owned request parameter**, not a pipeline constant. Each score is the instance's **softmax class probability after torchvision's per-class non-maximum suppression (NMS), not a calibrated** probability for your images. The default `0.75` is the value torchvision's visualization-utilities gallery uses to keep Mask R-CNN instances (shown there with the v1 weights, not re-derived for v2), and it is passed explicitly on every call. A mask pixel belongs to the instance when its predicted probability is at least `MASK_THRESHOLD` (0.5).

The scene is drawn, not photographed, so a miss here is a finding about renderings, not about photographs. The evaluation report records every miss with `box_iou = 0.0` and `mask_iou = 0.0`. COCO mean average precision needs a labelled image set; on one scene the verdict is `sample-sanity`.

**Predict before running:** all four drawn objects are COCO classes. How many will the model find at 0.75 with mask IoU ≥ 0.5, and which one do you expect a model trained on photographs to miss in a flat drawing?

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path

os.makedirs('outputs', exist_ok=True)
OUTPUTS = Path('outputs')

threshold = 0.75  # @param {type:"number"}

print({'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS,
       'label_slots': len(LABELS), 'unannotated_slots': LABELS.count('N/A'), 'DETECTION_THRESHOLD': DETECTION_THRESHOLD,
       'MASK_THRESHOLD': MASK_THRESHOLD})

scene, references = tutorial_scene()
buffer = io.BytesIO()
scene.save(buffer, format='PNG')
print({'sample_kind': 'synthetic', 'size': list(scene.size), 'sha256': hashlib.sha256(buffer.getvalue()).hexdigest()[:16],
       'references': {label: len(ref['boxes']) for label, ref in references.items()}})

input_manifest = validate_inputs(scene, threshold=threshold, names=['tutorial-scene'])
try:
    validate_inputs(scene, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'probe': 'threshold=1.5', 'rejected': str(exc)})
print({'verdict': input_manifest['verdict'], 'findings': input_manifest['findings'], 'inputs': input_manifest['inputs']})

coco_result = pipe.detect(scene, threshold=threshold)
for det in coco_result['detections']:
    print(f"{det['label']:>14s} {det['score']:.3f}  mask {det['mask_area']:>6d} px  [{', '.join(f'{v:.0f}' for v in det['box'])}]")

coco_report = evaluation_report(coco_result, references, sample_kind='synthetic')
print({'verdict': coco_report['verdict'], 'n_detections': coco_report['n_detections']})
for metric in coco_report['metrics']:
    print(f"  {metric['reference']:>18s}  box_iou {metric['box_iou']:.3f}  mask_iou {metric['mask_iou']:.3f}  same-label {metric['n_detected_same_label']}")
hits = sum(1 for m in coco_report['metrics'] if m['mask_iou'] >= 0.5)
print(f'{hits}/{len(coco_report["metrics"])} drawn objects matched at mask IoU >= 0.5')

overlay = np.asarray(scene).copy()
for mask in coco_result['masks']:
    overlay[mask] = (0.5 * overlay[mask] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
Image.fromarray(overlay)

**What to notice (Section 4):** the matched count, each instance's score, and the `box_iou` and `mask_iou` of each reference.

<details><summary>Check your reasoning</summary>In the recorded runs (Kaggle T4, 2026-09-25, and a local CPU run, 2026-10-03, identical to the printed precision) the model returned `stop sign` 0.9995, `clock` 0.9969 and `traffic light` 0.9953, each with mask IoU above 0.97 against the drawn mask (stop sign `box_iou` 0.944 / `mask_iou` 0.984), so 3 of the 4 drawn objects matched. The **sports ball was not detected** and is recorded with IoU 0.0. A flat orange disc carries little of what a photograph of a ball does (shading, seams, context), while an octagon with a word on it, a box of coloured lamps and a clock face look much like their photographs. A miss on a drawing is a finding about renderings, recorded, not hidden.</details>

## 5. Degenerate input probes: blank canvas and noise

Ask a segmentation model about structure-free input before trusting it. A model that returns confident instances on a blank canvas or on uniform noise will return them on empty real frames too. The cell counts instances on both images at the default threshold and at the evaluation threshold `EVAL_DETECTION_THRESHOLD` (0.05) that average precision is computed at.

**What to look for:** any instance above the default threshold on these two images is a false positive by construction. **Instances that appear only at 0.05 matter too.** Average precision (Section 9) ranks *every* instance scoring at least 0.05: a false positive ranked below every correct instance costs AP nothing, but one ranked above a correct instance lowers precision at that point of the ranking and so lowers AP. Low-scoring instances on nothing are harmless at 0.75 and are exactly what AP sees.

**Predict before running:** will either image return an instance at 0.75? Will noise return any at 0.05, and would they cover a small patch or most of the frame?

In [ ]:
degenerate = {}
for name, image in (('blank', blank_scene()), ('noise', noise_scene(0))):
    standard = pipe.detect(image, threshold=threshold)['detections']
    lenient = pipe.detect(image, threshold=EVAL_DETECTION_THRESHOLD)['detections']
    degenerate[name] = {
        'at_default_threshold': len(standard),
        'at_evaluation_threshold': len(lenient),
        'top': [(d['label'], round(d['score'], 3), d['mask_area']) for d in lenient[:3]],
    }
print(json.dumps(degenerate, indent=2))

**What to notice (Section 5):** the counts at 0.75 against the counts at 0.05, and the mask areas in `top`.

<details><summary>Check your reasoning</summary>The blank canvas returned nothing at either threshold. Noise returned nothing at 0.75 but **2 instances at 0.05** — `umbrella` 0.26 and `bear` 0.13, each covering most of the 640×480 frame (Kaggle T4, 2026-09-25, and a local CPU run alike). Neither would survive the default threshold, so the default output on noise is empty. On a labelled set, instances like these enter the AP ranking; scored at 0.13–0.26 they would sit below confident correct instances and cost little, but a model that scores clutter as high as its true objects loses AP for it.</details>

## 6. Labelled adaptation dataset and validation

Suppose your task needs sign classes that COCO does not have. `sign_dataset` draws a deterministic dataset of `N_IMAGES` images (default 40) over `SIGN_CLASSES`: `stop-sign`, `yield-sign` and `speed-limit-sign`. Every sign is drawn twice, once onto the image and once onto its own mask, so each reference mask is exact and each reference box is the tight box of its mask.

**Keep the two vocabularies apart.** COCO has `stop sign` (with a space). The adaptation vocabulary uses `stop-sign` (hyphenated), `yield-sign` and `speed-limit-sign`. They are different class identities, and the adapted model answers in the new names only.

`validate_dataset` checks every record before any model runs: the record keys, the image size ceilings, that every box is finite, non-empty and inside its image, that every mask has the image's shape, is boolean, is not empty and lies inside its box, and that every label is in the vocabulary. It returns a dataset manifest with the instance count per class and a finding for any class that has no instance. (The manifest also records the number of epochs; Section 8, where `EPOCHS` is set, validates again and keeps that manifest.)

**Predict before running:** with the default `N_IMAGES = 40`, roughly how many instances will the dataset hold, and will the three classes be balanced?

In [ ]:
N_IMAGES = 40  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}

records = sign_dataset(N_IMAGES, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, SIGN_CLASSES)
print(json.dumps({k: v for k, v in dataset_manifest.items() if k not in ('schema', 'epochs')}, indent=2))

preview = Image.new('RGB', (480, 320))
for index, record in enumerate(records[:3]):
    preview.paste(record['image'].resize((160, 160)), (160 * index, 0))
    union = np.any(np.stack(record['masks']), axis=0)
    preview.paste(Image.fromarray((union * 255).astype(np.uint8)).convert('RGB').resize((160, 160)), (160 * index, 160))
preview

**What to notice (Section 6):** `n_records`, `n_instances` and `instances_per_class`, and in the preview each image above its mask.

<details><summary>Check your reasoning</summary>At the defaults the recorded run validated 40 records with 68 instances: `stop-sign` 20, `yield-sign` 24, `speed-limit-sign` 24 — one to three signs per image, close to balanced. The manifest is built before any model runs, so a malformed record is refused before training, not discovered after it.</details>

## 7. Split, re-head, and measure the pre-adaptation baseline

The dataset is split at random into a training part (`1 - HOLDOUT`, default 75%) and a held-out part (`HOLDOUT`, default 25%); with the default 40 images that is 30 and 10. A random split is valid here because every image is drawn independently; records that share a photograph or a camera session must be split by that group instead. The held-out images are never shown to the optimizer, and no hyperparameter is selected on them.

`from_pretrained(class_names=SIGN_CLASSES)` loads the verified checkpoint and replaces only the two class-specific predictors with freshly initialised layers, seeded by `SEED`: the box predictor (`roi_heads.box_predictor`, now 3 classes plus background) and the mask predictor (`roi_heads.mask_predictor`, one 28×28 mask channel per class). This is **re-heading**. The backbone, the FPN, the RPN and the shared ROI layers keep their COCO weights. The cell wraps that call in `reset_to_pretrained()`, which Section 8 uses to start every fine-tune from this same re-headed model.

**The baseline is expected to be near zero.** Randomly initialised predictors have no information about the new classes, so this number is the floor the fine-tune has to beat, not a property of Mask R-CNN.

**Predict before running:** the COCO model knew `stop sign`. Will the baseline score anything on `stop-sign`, and on the two new shapes?

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_dataset(records, train_fraction=1.0 - HOLDOUT, seed=SEED)
overlap = {r['id'] for r in train_records} & {r['id'] for r in held_out}
assert not overlap, f'split leaked records: {sorted(overlap)}'
print({'train': len(train_records), 'held_out': len(held_out),
       'train_instances': sum(len(r['boxes']) for r in train_records),
       'held_out_instances': sum(len(r['boxes']) for r in held_out)})

def reset_to_pretrained():
    """A fresh re-headed model: the verified checkpoint with new box and mask predictors initialised under SEED."""
    return MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SIGN_CLASSES, seed=SEED)

adapter = reset_to_pretrained()
print({'class_names': list(adapter.class_names), 'device': adapter.device, 'reinitialised': list(adapter.reinitialised)})

baseline = adapter.evaluate(held_out)
print(json.dumps({key: round(baseline[key], 4) for key in ('ap', 'ap50', 'mask_ap', 'mask_ap50')}
                 | {'per_class_ap50': {k: round(v, 4) for k, v in baseline['per_class_ap50'].items()}, 'n_references': baseline['n_references']}, indent=2))

**What to notice (Section 7):** the split sizes, the two re-initialised predictors, and the baseline per class.

<details><summary>Check your reasoning</summary>The recorded run split 30 / 10 images (51 / 17 instances) and measured a baseline box AP50 of 0.1782 and mask AP of 0.0: AP50 0.5347 on `stop-sign` and 0.0 on the two new shapes. The predictors are random, so whatever the baseline finds is chance passed through the shared COCO layers (why it is `stop-sign` that scores is a guess: the COCO model knew stop signs); no mask matched at all.Near zero is the expected floor.</details>

## 8. Bounded fine-tuning

This cell runs the real adaptation step in this runtime. It is gradient fine-tuning with Mask R-CNN's own training losses, which torchvision returns when the model is called with targets: RPN objectness and RPN box regression, ROI classification and ROI box regression, and a per-pixel binary cross-entropy on the 28×28 mask of each positive region. The optimised loss is their sum.

- **Every training setting is in this cell:** `EPOCHS` (default 10), `LEARNING_RATE` (0.005, the value of torchvision's detection fine-tuning tutorial, with SGD momentum 0.9 and weight decay 0.0005), `BATCH_SIZE` (2) and `FREEZE_BACKBONE`; float32, seed `SEED`. The cell re-validates the dataset with `EPOCHS`, so the manifest records the schedule actually run.
- **The ResNet-50 body is frozen.** With the default `FREEZE_BACKBONE = True`, parameters under `backbone.body.` keep their COCO values, and its BatchNorm layers are held in evaluation mode so their running statistics do not drift either. The FPN, the RPN and the ROI heads are trained. The cell prints the trainable and total parameter counts. (Unfreezing it is the Section 14 activity.)
- **Adaptation always starts from the re-headed model.** `finetune` changes the model in place, and it refuses a model that is already adapted. If you re-run this cell after a fine-tune, it first calls `reset_to_pretrained()` — the same fresh predictors that Section 7's baseline measured — so a changed setting is compared from the same starting point, and the printed run record describes exactly the schedule that produced the model.

**Read the loss as optimisation evidence only.** A falling loss says the optimizer is fitting the training images; the held-out average precision in the next section is the task evidence.

**Predict before running:** will the first epoch's loss be above or below 1.0, and by what factor will it fall over the default 10 epochs?

In [ ]:
import time

EPOCHS = 10  # @param {type:"integer"}
LEARNING_RATE = 0.005  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
FREEZE_BACKBONE = True  # @param {type:"boolean"}

dataset_manifest = validate_dataset(records, SIGN_CLASSES, epochs=EPOCHS)
if adapter.adapted:
    # A re-run: start again from the re-headed base (same SEED, same predictors as the Section 7 baseline).
    adapter = reset_to_pretrained()
    print({'rebuilt_from_verified_snapshot': True, 'adapted': adapter.adapted})
train_started = time.perf_counter()
run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_backbone=FREEZE_BACKBONE,
    progress=lambda row: print(f"epoch {row['epoch']}/{row['epochs']}  loss {row['loss']:.4f}"),
)
train_seconds = round(time.perf_counter() - train_started, 1)
print(json.dumps({**{key: run[key] for key in ('freeze_backbone', 'trainable_parameters', 'total_parameters', 'epochs',
                                             'batch_size', 'learning_rate', 'optimizer', 'precision', 'device')},
                  'started_from': 'the re-headed checkpoint (fresh predictors, seed ' + str(SEED) + ')',
                  'first_epoch_loss': round(run['epoch_losses'][0], 4), 'final_loss': round(run['final_loss'], 4),
                  'train_seconds': train_seconds}, indent=2))

**What to notice (Section 8):** the epoch-1 loss, how fast it falls, and the trainable share of the parameters.

<details><summary>Check your reasoning</summary>The recorded run (Kaggle T4, defaults) started at 0.9822 — just below 1.0 — and ended at 0.0863 after 10 epochs (0.9822, 0.3973, 0.2616, 0.1543, 0.1247, 0.1093, 0.1026, 0.0903, 0.0854, 0.0863): about a tenfold fall, levelling off after epoch 7. With the ResNet-50 body frozen, 22,383,143 of 45,891,175 parameters were trainable. A loss this low says the training images are fitted; it does not yet say anything about images the optimizer never saw.</details>

## 9. Evaluate on the held-out split, and what a 1.0 row means

`evaluate` re-runs on the same held-out images with the same thresholds as the baseline, so the two rows are comparable. It scores every instance twice: once by box IoU (`ap`, `ap50`, `ap75`) and once by mask IoU (`mask_ap`, `mask_ap50`, `mask_ap75`). `ap50` is average precision at IoU 0.50; `ap` averages AP over the ten IoU thresholds 0.50–0.95, so it also rewards tight boxes and masks. A model can find every sign with a good box and still draw a poor mask, which only the mask rows show. AP ranks every instance scoring at least `EVAL_DETECTION_THRESHOLD` (0.05), so it measures ranking, not what a 0.75 threshold returns. These are tutorial metrics from one pass over the held-out images (10 at the defaults), with no dispersion estimate.

**A 1.0 row is a ceiling.** AP cannot exceed 1.0: a row reading 1.0000 means every held-out instance was found, at every IoU threshold the row averages, and ranked above every false positive. On this drawn task the adapted model may reach that ceiling — the recorded run did on every mask row. A saturated row cannot show a further gain, so it cannot rank two settings: two runs that both read 1.0 may still differ in how much margin they have. It also says the task is easy for this model at this size, not that the model segments real signs perfectly. To compare settings, read the rows that are still below 1.0. The run-history table at the end of the cell keeps one row per evaluation, so a Section 14 re-run prints side by side with the default run.

**Predict before running:** which rows will reach 1.0 after the default fine-tune, and which will stay below it? Why might box `ap` stay below mask `ap` on these drawn signs?

In [ ]:
adapted = adapter.evaluate(held_out)
print(f"{'metric':<10s} {'baseline':>10s} {'adapted':>10s} {'change':>10s}")
for key in ('ap', 'ap50', 'ap75', 'mask_ap', 'mask_ap50', 'mask_ap75'):
    print(f"{key:<10s} {baseline[key]:>10.4f} {adapted[key]:>10.4f} {adapted[key] - baseline[key]:>+10.4f}")
print('per-class mask AP50:', {k: round(v, 4) for k, v in adapted['per_class_mask_ap50'].items()})
saturated = [key for key in ('ap', 'ap50', 'ap75', 'mask_ap', 'mask_ap50', 'mask_ap75') if adapted[key] >= 1.0 - 1e-9]
headroom = [key for key in ('ap', 'ap50', 'ap75', 'mask_ap', 'mask_ap50', 'mask_ap75') if key not in saturated]
print({'at_ceiling_1.0': saturated, 'below_ceiling': headroom, 'evaluation_threshold': EVAL_DETECTION_THRESHOLD})

RUN_HISTORY = globals().get('RUN_HISTORY') or []
RUN_HISTORY.append({'freeze_backbone': run['freeze_backbone'], 'epochs': run['epochs'], 'learning_rate': run['learning_rate'],
                    'first_epoch_loss': round(run['epoch_losses'][0], 4), 'final_loss': round(run['final_loss'], 4),
                    'ap': round(adapted['ap'], 4), 'ap75': round(adapted['ap75'], 4), 'mask_ap': round(adapted['mask_ap'], 4),
                    'mask_ap75': round(adapted['mask_ap75'], 4), 'train_seconds': train_seconds})
print('run history (one row per evaluation; row 0 is the first run of this session):')
for index, row in enumerate(RUN_HISTORY):
    print(index, row)

**What to notice (Section 9):** which rows reached 1.0, which did not, and the `below_ceiling` list.

<details><summary>Check your reasoning</summary>In the recorded run (Kaggle T4, defaults) every held-out row moved from near zero to the ceiling except one: `ap50` 0.1782 → 1.0, `ap75` 0.0 → 1.0, `mask_ap`, `mask_ap50` and `mask_ap75` 0.0 → 1.0, and every per-class mask AP50 1.0; only box `ap` stayed below it, 0.0347 → 0.8693. Since box `ap50` and `ap75` are 1.0, the shortfall is at the strictest box IoU thresholds (above 0.75): some boxes are not tight to the last few pixels, while the masks are. Read the saturated rows as *the drawn task is solved at this size*, not as *perfect segmentation*: ten held-out images drawn by the same generator as the training images cannot show how far the model is from failing, nor anything about photographs. That is why Section 14 compares settings on box `ap`, the run history's one column with headroom here.</details>

## 10. Inference on unseen images

Three new images come from a seed the dataset never used (`NEW_DATA_SEED`, default 99). The adapted pipeline runs at the same `threshold` as the COCO model (0.75), and each reference instance is compared with the best same-label instance by box IoU and by mask IoU.

**Predict before running:** how many of the reference signs will be found at 0.75, and will the mask IoUs be higher or lower than the box IoUs?

In [ ]:
NEW_DATA_SEED = 99  # @param {type:"integer"}

new_records = sign_dataset(3, seed=NEW_DATA_SEED)
new_data_rows = []
for record in new_records:
    out = adapter.detect(record['image'], threshold=threshold)
    box_ious, mask_ious = [], []
    for box, label, mask in zip(record['boxes'], record['labels'], record['masks'], strict=True):
        same = [i for i, d in enumerate(out['detections']) if d['label'] == label]
        box_ious.append(round(max((box_iou(out['detections'][i]['box'], box) for i in same), default=0.0), 3))
        mask_ious.append(round(max((mask_iou(out['masks'][i], mask) for i in same), default=0.0), 3))
    row = {'id': record['id'], 'truth': record['labels'],
           'detections': [(d['label'], round(d['score'], 3)) for d in out['detections']],
           'same_label_box_iou': box_ious, 'same_label_mask_iou': mask_ious}
    new_data_rows.append(row)
    print(json.dumps(row))
n_new = sum(len(row['truth']) for row in new_data_rows)
found = sum(1 for row in new_data_rows for value in row['same_label_mask_iou'] if value >= 0.5)
print(f'{found}/{n_new} reference instances found at threshold {threshold} (same label, mask IoU >= 0.5)')

**What to notice (Section 10):** the found count, each row's `detections`, and box IoU against mask IoU.

<details><summary>Check your reasoning</summary>The recorded run (Kaggle T4, defaults) returned 6 instances for the 6 reference signs on the three unseen images, every label correct, with same-label `box_iou` 0.926–0.944 and `mask_iou` 0.964–0.980: the masks fit more tightly than the boxes, as the held-out `ap` against `mask_ap` suggested. These images come from the same drawing code as the training set, so this is a check that the adapted model generalises to new *drawings*, not to photographs.</details>

## 11. Adapter export, fresh reload, and equivalence check

`save_artifact` writes `outputs/maskrcnn_adapter.safetensors`: every tensor the fine-tune could change, plus a metadata header naming the base model, its pinned checkpoint SHA-256, the class names and the frozen prefixes. The frozen ResNet-50 body is left out because it equals the verified base checkpoint.

`load_artifact` then builds a **fresh** pipeline from the verified base checkpoint, loads the adapter tensors onto it, and refuses an adapter whose format, base identity, base digest or tensor set does not fit. The cell compares the reloaded instances with the in-memory model's on an unseen image, with a stated tolerance: loading succeeding is not the check, reproducing the boxes, scores and masks is. The comparison runs at the evaluation threshold 0.05; if nothing passes it, it compares the 10 top-scoring instances, and if the model returns no instance at all it compares every weight tensor, so the check is never empty. The same `reload_equivalence` helper checks the BYOD adapter in Section 13.

**Predict before running:** why compare instances after the reload instead of only checking that the file loads?

In [ ]:
artifact_path = OUTPUTS / 'maskrcnn_adapter.safetensors'
descriptor = adapter.save_artifact(artifact_path, notes='Mask R-CNN ResNet-50 FPN v2 sign adaptation tutorial adapter')
print(json.dumps(descriptor, indent=2))

reloaded = MaskRcnnPipeline.load_artifact(artifact_path, weights_dir=WEIGHTS_DIR)
print({'reloaded_source': reloaded.source, 'adapted': reloaded.adapted, 'class_names': list(reloaded.class_names)})

TOLERANCE = 1e-3
MASK_AGREEMENT = 0.99

def reload_equivalence(first, second, image, tolerance=TOLERANCE):
    """Both pipelines must return the same instances on `image` (label, box and score within `tolerance`, masks at IoU >= MASK_AGREEMENT)."""
    at, limit = EVAL_DETECTION_THRESHOLD, None
    if not first.detect(image, threshold=at)['detections']:  # nothing passes 0.05 (e.g. a short BYOD run)
        at, limit = 0.0, 10  # compare the 10 top-scoring instances instead, so the check is never empty
    out_orig = first.detect(image, threshold=at)
    out_reloaded = second.detect(image, threshold=at)
    pairs = list(zip(out_orig['detections'], out_reloaded['detections'], out_orig['masks'], out_reloaded['masks']))[:limit]
    if len(out_orig['detections'][:limit]) != len(out_reloaded['detections'][:limit]):
        raise RuntimeError(f"the reloaded adapter returned {len(out_reloaded['detections'])} instances, the in-memory model {len(out_orig['detections'])}: the export is not faithful")
    for d1, d2, m1, m2 in pairs:
        if d1['label'] != d2['label'] or not np.allclose(d1['box'], d2['box'], atol=tolerance) or abs(d1['score'] - d2['score']) > tolerance:
            raise RuntimeError(f'reloaded instance {d2} differs from {d1} beyond tolerance {tolerance}: the export is not faithful')
        if (m1.any() or m2.any()) and mask_iou(m1, m2) < MASK_AGREEMENT:
            raise RuntimeError(f"reloaded mask of {d1['label']} agrees with the original at IoU {mask_iou(m1, m2):.4f} < {MASK_AGREEMENT}")
    if not pairs:  # no instance at all, even unthresholded (e.g. a barely trained BYOD model): compare the weights instead
        state_orig, state_reloaded = first.model.state_dict(), second.model.state_dict()
        differing = [k for k in state_orig if not torch.allclose(state_orig[k].float().cpu(), state_reloaded[k].float().cpu(), atol=tolerance)]
        if differing:
            raise RuntimeError(f'the reloaded adapter differs from the in-memory model in {len(differing)} tensors, e.g. {differing[:3]}: the export is not faithful')
        return {'instances_compared': 0, 'tensors_compared': len(state_orig), 'tolerance': tolerance, 'equivalent': True}
    return {'instances_compared': len(pairs), 'threshold': at, 'tolerance': tolerance, 'mask_iou_at_least': MASK_AGREEMENT, 'equivalent': True}

reload_check = reload_equivalence(adapter, reloaded, new_records[0]['image'])
print(reload_check)

**What to notice (Section 11):** `instances_compared` (never 0) and `equivalent`.

<details><summary>Check your reasoning</summary>A file can load and still hold the wrong weights — an earlier adapter, a different base checkpoint, a tensor saved before training finished. Comparing what the reloaded model *does* catches all of those. The recorded run wrote an 89,577,652-byte adapter of 114 tensors and found the instances equivalent at tolerance 0.001 and mask IoU ≥ 0.99; any difference stops the notebook, because it is a contract failure, not a quality result.</details>

## 12. Write machine-readable outputs and provenance

The cell writes:
- `outputs/maskrcnn_instance_segmentation_input_manifest.json`
- `outputs/maskrcnn_instance_segmentation_evaluation_report.json`
- `outputs/maskrcnn_instance_segmentation_result.json` (identity, runtime versions, device, dataset manifest, split, baseline and adapted metrics, the saturated and below-ceiling rows, the run history, fine-tuning configuration with its training time, new-data rows, adapter descriptor and reload check)
- `outputs/maskrcnn_instance_segmentation_detections.csv` (one row per instance, with its mask area)
- `outputs/maskrcnn_instance_segmentation_masks.npz` (the demonstration scene's boolean masks)
- `outputs/maskrcnn_instance_segmentation_annotated.png`
- `outputs/maskrcnn_adapter.safetensors` (written in Section 11)

In [ ]:
import csv
from PIL import ImageDraw

with open(OUTPUTS / 'maskrcnn_instance_segmentation_input_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(input_manifest, f, indent=2)

with open(OUTPUTS / 'maskrcnn_instance_segmentation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(coco_report, f, indent=2)

np.savez_compressed(OUTPUTS / 'maskrcnn_instance_segmentation_masks.npz', *coco_result['masks'])

annotated_pixels = np.asarray(scene).copy()
for mask in coco_result['masks']:
    annotated_pixels[mask] = (0.5 * annotated_pixels[mask] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
annotated = Image.fromarray(annotated_pixels)
draw = ImageDraw.Draw(annotated)
for det in coco_result['detections']:
    x0, y0, x1, y1 = det['box']
    draw.rectangle([x0, y0, x1, y1], outline='red', width=3)
    draw.text((x0 + 4, y0 + 4), f"{det['label']} {det['score']:.2f}", fill='red')
annotated.save(OUTPUTS / 'maskrcnn_instance_segmentation_annotated.png')

with open(OUTPUTS / 'maskrcnn_instance_segmentation_detections.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1', 'mask_area'])
    for rank, d in enumerate(coco_result['detections']):
        writer.writerow(['tutorial-scene', rank, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box']), d['mask_area']])
    for row, record in zip(new_data_rows, new_records, strict=True):
        for rank, d in enumerate(adapter.detect(record['image'], threshold=threshold)['detections']):
            writer.writerow([row['id'], rank, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box']), d['mask_area']])

result_export = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'weights_url': WEIGHTS_URL,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                'cuda': torch.cuda.is_available()},
    'device': pipe.device,
    'threshold': threshold,
    'mask_threshold': MASK_THRESHOLD,
    'evaluation_threshold': EVAL_DETECTION_THRESHOLD,
    'coco_detections': coco_result['detections'],
    'degenerate_probes': degenerate,
    'adaptation': {
        'dataset': {k: v for k, v in dataset_manifest.items() if k != 'schema'},
        'dataset_seed': DATASET_SEED,
        'split': {'train': len(train_records), 'held_out': len(held_out), 'seed': SEED, 'holdout': HOLDOUT},
        'finetune': {**run, 'train_seconds': train_seconds},
        'baseline': baseline,
        'adapted': adapted,
        'at_ceiling': saturated,
        'below_ceiling': headroom,
        'run_history': RUN_HISTORY,
        'new_data': new_data_rows,
        'artifact': descriptor,
        'reload_check': reload_check,
    },
}
with open(OUTPUTS / 'maskrcnn_instance_segmentation_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_export, f, indent=2, default=str)

for p in sorted(OUTPUTS.iterdir()):
    if p.is_file():
        print(f'  {p.name:<44s} {p.stat().st_size:>12,d} bytes')

## 13. Optional: Bring Your Own Data (BYOD)

Both branches are off by default, so `Run all` never stops here. Before you turn one on, read the contract:

- **Image branch** (`USE_BYOD_IMAGE`): one image file Pillow can open, each side between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px. It runs through `validate_inputs`, `detect` and `evaluation_report` — the report is `not-measurable`, because no reference instances come with it — and writes `outputs/byod_maskrcnn_input_manifest.json`. Upload exactly one image; every upload replaces the previous one.
- **Dataset branch** (`USE_BYOD_DATASET`): a directory with `annotations.json` (a list of `{'file', 'boxes', 'labels', 'masks'}` objects: **xyxy pixel boxes** — not coordinates normalised to 0..1 — and one mask PNG name per instance) and the image and mask files it names, at least 2 records and at most 5,000. File names must stay inside the directory. **Through the upload dialog, upload `annotations.json`, the images and the masks as flat files (no folders).** `BYOD_CLASS_NAMES` is a comma-separated vocabulary; leave it empty to use the sorted set of labels in the annotations. The branch runs the same validate → split → baseline → fine-tune → evaluate → export → reload stages as the sample with the Section 7 and 8 settings, compares the reloaded adapter's instances with the in-memory model's, and writes `outputs/byod_maskrcnn_result.json` (annotation digest, dataset manifest, split, fine-tune configuration, baseline and adapted metrics, adapter descriptor and reload check) and `outputs/byod_maskrcnn_detections.csv` (held-out instances at the evaluation threshold).

Set `BYOD_IMAGE_PATH` (a file) or `BYOD_DATASET_DIR` (a directory) to read from a mounted or local location; leave them empty on Colab to get an upload dialog instead. Uploaded files are written under `outputs/byod/` in this runtime and are not sent anywhere else. **Every refusal names the file or the rule and what to do:** a path that is not a file or directory, a runtime without an upload dialog, a cancelled upload, a file Pillow cannot read, a mask file named in `annotations.json` but missing, and boxes that look normalised. The first lines of the cell show three of those refusals on purpose. After changing a field here, re-run this cell only.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE_PATH = ""  # @param {type:"string"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_DATASET_DIR = ""  # @param {type:"string"}
BYOD_CLASS_NAMES = ""  # @param {type:"string"}

import shutil

BYOD_DIR = OUTPUTS / 'byod'

def _open_byod_image(path):
    """Open one BYOD image fully, or raise a ValueError that names the file and the fix."""
    try:
        with Image.open(path) as handle:
            return handle.convert('RGB')
    except (OSError, SyntaxError, ValueError) as exc:  # PIL.UnidentifiedImageError is an OSError
        raise ValueError(f'{Path(path).name} is not an image Pillow can read ({type(exc).__name__}): use a PNG or JPEG file, each side {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px') from exc

def _existing(path_text, field, kind):
    """The path in a location field, checked before anything is opened."""
    path = Path(path_text.strip()).expanduser()
    if not (path.is_dir() if kind == 'directory' else path.is_file()):
        expected = 'the directory holding annotations.json' if kind == 'directory' else 'one image file'
        raise ValueError(f'{field} {str(path)!r} is not a {kind}: set it to the path of {expected} in this runtime')
    return path

def _upload_into(target, field):
    """Colab upload dialog into a fresh `target`: an earlier upload never mixes with this one."""
    try:
        from google.colab import files  # type: ignore[import-not-found]
    except ImportError:
        raise RuntimeError(f'{field} is empty and this runtime has no Colab upload dialog: set {field} to a path in this runtime') from None
    shutil.rmtree(target, ignore_errors=True)
    target.mkdir(parents=True)
    uploaded = files.upload() or {}
    if not uploaded:
        raise ValueError(f'the upload was cancelled or empty: run this cell again and choose the files, or set {field}')
    for name, data in uploaded.items():
        (target / Path(name).name).write_bytes(data)
    return target

_probe_dir = OUTPUTS / 'byod_refusal_probes'
_probe_dir.mkdir(parents=True, exist_ok=True)
(_probe_dir / 'notes.png').write_text('this is text, not an image', encoding='utf-8')
_probe_image = blank_scene()
_probe_mask = np.zeros((_probe_image.height, _probe_image.width), dtype=bool)
_probe_mask[40:80, 40:80] = True
for desc, probe in (
    ('a file that is not an image', lambda: _open_byod_image(_probe_dir / 'notes.png')),
    ('an empty mask', lambda: validate_dataset([{'image': _probe_image, 'boxes': [[0, 0, 10, 10]], 'labels': [SIGN_CLASSES[0]],
                                               'masks': [np.zeros_like(_probe_mask)]}], SIGN_CLASSES)),
    ('a normalised box', lambda: validate_dataset([{'image': _probe_image, 'boxes': [[0.06, 0.08, 0.13, 0.17]], 'labels': [SIGN_CLASSES[0]],
                                                  'masks': [_probe_mask]}], SIGN_CLASSES)),
):
    try:
        probe()
    except (TypeError, ValueError) as exc:
        print(f'refused as expected: {desc} -> {type(exc).__name__}: {exc}')

if USE_BYOD_IMAGE:
    if BYOD_IMAGE_PATH.strip():
        image_path = _existing(BYOD_IMAGE_PATH, 'BYOD_IMAGE_PATH', 'file')
    else:
        uploaded_images = sorted(p for p in _upload_into(BYOD_DIR / 'image', 'BYOD_IMAGE_PATH').iterdir() if p.is_file())
        if len(uploaded_images) != 1:
            raise ValueError(f'upload exactly one image for the image branch, got {len(uploaded_images)}: run this cell again')
        image_path = uploaded_images[0]
    byod_image = _open_byod_image(image_path)
    byod_input_manifest = validate_inputs(byod_image, threshold=threshold, names=[image_path.name])
    print(byod_input_manifest['verdict'])
    byod_result = pipe.detect(byod_image, threshold=threshold)
    for det in byod_result['detections'][:20]:
        print(f"{det['label']:>16s} {det['score']:.3f}  mask {det['mask_area']:>7d} px  [{', '.join(f'{v:.0f}' for v in det['box'])}]")
    byod_input_manifest['n_detections'] = len(byod_result['detections'])
    byod_input_manifest['report_verdict'] = evaluation_report(byod_result, None, sample_kind='byod')['verdict']
    byod_input_manifest['sha256'] = hashlib.sha256(image_path.read_bytes()).hexdigest()
    with open(OUTPUTS / 'byod_maskrcnn_input_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(byod_input_manifest, f, indent=2, default=str)
    print(byod_input_manifest['report_verdict'], '-> wrote byod_maskrcnn_input_manifest.json')
else:
    print('BYOD image branch is off; set USE_BYOD_IMAGE = True to run segmentation on your own image.')

if USE_BYOD_DATASET:
    if BYOD_DATASET_DIR.strip():
        dataset_dir = _existing(BYOD_DATASET_DIR, 'BYOD_DATASET_DIR', 'directory')
    else:
        dataset_dir = _upload_into(BYOD_DIR / 'dataset', 'BYOD_DATASET_DIR')
    byod_records = read_detection_records(dataset_dir)
    byod_names = [n.strip() for n in BYOD_CLASS_NAMES.split(',') if n.strip()] or sorted({l for r in byod_records for l in r['labels']})
    byod_manifest = validate_dataset(byod_records, byod_names, epochs=EPOCHS)
    print(json.dumps({k: v for k, v in byod_manifest.items() if k != 'schema'}, indent=2))
    byod_train, byod_held = split_dataset(byod_records, train_fraction=1.0 - HOLDOUT, seed=SEED)
    byod_pipe = MaskRcnnPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=byod_names, seed=SEED)
    byod_baseline = byod_pipe.evaluate(byod_held)
    byod_run = byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, freeze_backbone=FREEZE_BACKBONE)
    byod_adapted = byod_pipe.evaluate(byod_held)
    print({key: (round(byod_baseline[key], 4), round(byod_adapted[key], 4)) for key in ('ap', 'ap50', 'mask_ap', 'mask_ap50')})
    byod_artifact = OUTPUTS / 'byod_maskrcnn_adapter.safetensors'
    byod_descriptor = byod_pipe.save_artifact(byod_artifact, notes='BYOD adaptation adapter')
    byod_reloaded = MaskRcnnPipeline.load_artifact(byod_artifact, weights_dir=WEIGHTS_DIR)
    byod_reload_check = reload_equivalence(byod_pipe, byod_reloaded, byod_held[0]['image'])
    print('BYOD adapter exported, reloaded and compared:', byod_descriptor['sha256'][:16], byod_reload_check)
    with open(OUTPUTS / 'byod_maskrcnn_detections.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1', 'mask_area'])
        for record in byod_held:
            for rank, d in enumerate(byod_pipe.detect(record['image'], threshold=EVAL_DETECTION_THRESHOLD)['detections']):
                writer.writerow([record['id'], rank, d['label'], f"{d['score']:.4f}", *(f"{v:.1f}" for v in d['box']), d['mask_area']])
    byod_export = {
        'notebook_source': NOTEBOOK_SOURCE, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
        'annotations_sha256': hashlib.sha256((dataset_dir / 'annotations.json').read_bytes()).hexdigest(),
        'dataset': {k: v for k, v in byod_manifest.items() if k != 'schema'},
        'split': {'train': [r['id'] for r in byod_train], 'held_out': [r['id'] for r in byod_held], 'seed': SEED, 'holdout': HOLDOUT},
        'finetune': byod_run, 'baseline': byod_baseline, 'adapted': byod_adapted,
        'evaluation_threshold': EVAL_DETECTION_THRESHOLD, 'artifact': byod_descriptor, 'reload_check': byod_reload_check,
    }
    with open(OUTPUTS / 'byod_maskrcnn_result.json', 'w', encoding='utf-8') as f:
        json.dump(byod_export, f, indent=2, default=str)
    print('wrote', [p.name for p in sorted(OUTPUTS.glob('byod_maskrcnn_*'))])
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True to adapt Mask R-CNN on your own labelled images.')

## 14. Your turn — change one thing: unfreeze the backbone

Optional; **Predict → Change → Run → Observe → Explain**. It re-runs the fine-tune with the ResNet-50 body trainable, which takes longer than Section 8 did (about twice as many parameters get gradients). Compare on box `ap` (AP@[.50:.95]), the run history's column that was still below 1.0 in the recorded run, and on the losses and `train_seconds`; a row that reads 1.0 in both runs (Section 9) cannot tell them apart.

1. **Predict:** with `FREEZE_BACKBONE = False`, will held-out box `ap` go up or down against the default run, and will training take longer? Write your guess next to row 0 of the Section 9 run history.
2. **Change:** in Section 8 set `FREEZE_BACKBONE = False`. Change nothing else.
3. **Run:** select the Section 8 cell and choose **Runtime → Run after**. Section 8 calls `reset_to_pretrained()` before training (it prints `rebuilt_from_verified_snapshot`), Section 9 adds a row to the run history, and Sections 10–12 re-run on the new adapter.
4. **Observe:** Section 8 prints `rebuilt_from_verified_snapshot`, a trainable count of every parameter, and an epoch-1 loss close to the default run's (both runs start from the same re-headed model) — not the much lower loss that continuing from the adapted model would give. In Section 9, compare rows 0 and 1 of the run history: `ap`, `ap75`, `mask_ap`, `mask_ap75`, `final_loss` and `train_seconds`.
5. **Explain:** in one sentence, why is a change in box `ap` informative here while an unchanged `mask_ap` of 1.0 is not?

<details><summary>Check your reasoning</summary>In a local CPU check of this revision (2026-10-06, real weights, reduced to `N_IMAGES = 8` and `EPOCHS = 1` for time) the re-run after a default run printed `rebuilt_from_verified_snapshot`; its box and mask predictors on entry to `finetune` were equal, tensor for tensor, to a fresh `from_pretrained(class_names=SIGN_CLASSES, seed=0)`; all 45,891,175 parameters were trained; its first-epoch loss was 2.2127, a fresh start next to the frozen run's 2.1416; and the run record said `epochs: 1`, the schedule actually applied. Before this fix (the 2026-10-02 review) the same re-run continued training the adapted model while its record said one epoch. At that reduced size neither run is near the ceiling (box `ap` 0.1966 frozen and 0.4770 unfrozen, on 2 held-out images), which is far too small a test to rank the settings. The full 10-epoch unfrozen comparison on a GPU is not recorded for this revision, so your run history is the measurement. A `mask_ap` of 1.0 in both rows says both runs reached the ceiling on these ten drawn images; it cannot say which has more margin. Box `ap` below 1.0 can still move. One run on 10 held-out images carries no dispersion estimate, so read differences of a few hundredths as unresolved, not as a ranking of the two settings.</details>

## Interpretation and limits

Start from your own run: the Section 9 table and its `at_ceiling_1.0` list, the Section 10 found count and the Section 11 reload check. Then read the reference answer.

<details><summary>Reference answer from the recorded run</summary>The pretrained COCO model found 3 of the 4 drawn COCO objects at 0.75 (the sports ball was missed) and returned nothing on blank or noise images at 0.75, though noise produced 2 large instances at 0.05. Re-heading onto three new sign classes started from a baseline box AP50 of 0.18 and mask AP of 0.0; a 10-epoch fine-tune with the body frozen took every held-out mask row and box AP50 and AP75 to the ceiling of 1.0 and box AP@[.50:.95] to 0.87 (Kaggle T4, 2026-09-25). The pretrained model already solves most of this drawn task once its predictors are re-headed, so the lesson is the workflow — validate, split, baseline, fine-tune, evaluate, export, reload and compare — not the size of the gain. The adapted model found all 6 signs on the three unseen drawings, and the adapter reloaded to the same instances within 1e-3.</details>

**What this notebook established, in this runtime.** The pinned torchvision checkpoint for `torchvision/maskrcnn_resnet50_fpn_v2` was verified against a committed SHA-256 manifest before loading. The pretrained model was run on a drawn scene and on two structure-free probes, and its boxes and masks were compared with the drawn references by IoU. A three-class sign vocabulary that COCO does not contain was then adapted by replacing the box and mask predictors, training the FPN, RPN and ROI heads with the ResNet-50 body frozen, and scoring the held-out split with box and mask average precision before and after. The adapter was exported, reloaded onto a fresh base model and checked against the in-memory instances.

**A task at its ceiling.** When the adapted rows read 1.0, the held-out images no longer separate good settings from better ones. That is a property of this drawn task and its size, stated here rather than hidden: the signs are drawn in fixed colours and shapes by the same code as the training images. A harder test needs harder data — more varied drawings, photographs, or your own labelled images through Section 13 — not a different metric chosen after the fact.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set and the pinned checkpoint together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model generalises beyond the synthetic images it was fitted to. The held-out AP is measured on a few drawn images (10 at the defaults) and carries no dispersion estimate. The scores are not calibrated probabilities.

**Reproducibility.** Seeds are form fields (`DATASET_SEED`, `SEED`, `NEW_DATA_SEED`), the run is float32 with no data augmentation, and the new predictors are initialised under `SEED`. GPU kernels (ROI Align in particular) are not forced to be deterministic, so repeated GPU runs can differ in the last digits of the loss and the scores.

**More experiments (optional).** Each one names the cell to change and where to re-run from; a re-run of Section 8 always starts from the re-headed model, and Section 9's run history keeps the earlier rows.

- **Find where the ceiling starts:** set `EPOCHS = 1` (then 2, then 3) in Section 8, select Section 8 and choose **Runtime → Run after**. Watch at which epoch count the mask rows reach 1.0 and whether box `ap` is still climbing after that.
- **A different learning rate:** set `LEARNING_RATE = 0.001` in Section 8 and **Run after** from Section 8.
- **A smaller dataset:** set `N_IMAGES = 12` in Section 6, select Section 6 and choose **Run after**: fewer held-out images make every AP step coarser.
- **Your own labelled images:** turn on `USE_BYOD_DATASET` in Section 13 and re-run that cell only.

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment is built from manylinux wheels; use Google Colab, Kaggle or a Linux Jupyter server.
- **Section 1 fails while downloading.** The `uv` wheel, the managed Python and the locked packages come from PyPI and python-build-standalone; run the cell again. A size or SHA-256 mismatch is refused on purpose — if it repeats, the download is being altered.
- **A restart prompt.** This notebook never needs one: nothing is installed into the kernel. If the isolated process exits (usually out of memory), restart the session and choose **Run all**.
- **Section 3 stops on a download or a digest mismatch.** The checkpoint is fetched from download.pytorch.org and re-hashed; a mismatch is never loaded. Run the cell again; if it repeats, delete `weights/` and run from Section 3.
- **Out of memory in Section 8 or 14.** Lower `BATCH_SIZE` to 1 in Section 8, or keep `FREEZE_BACKBONE = True`, and **Run after** from Section 8. On a CPU runtime the fine-tune is slow, not broken (see the estimate under Prerequisites).
- **"Cannot fine-tune: this pipeline was already adapted".** Something called `finetune` on a model that was already trained; Section 8 avoids this by calling `reset_to_pretrained()`. If you edited the cells, re-run from Section 7.
- **Every adapted row reads 1.0.** That is the recorded result on this drawn task (Section 9 explains it), not an error; compare settings on the rows listed under `below_ceiling`.
- **BYOD is refused.** Each refusal names the file or the rule and the fix: a path that is not a file or directory, no upload dialog outside Colab, a cancelled upload, a file Pillow cannot read, a mask file missing from the folder (usually uploaded in a sub-folder; upload flat files), boxes that look normalised to 0..1 (use pixel coordinates), a mask outside its box, an unknown label, or fewer than 2 records. Fix the data, then re-run Section 13.

## Glossary

- **Instance segmentation:** one box, one class and one pixel mask per object, so two touching objects of the same class stay separate.
- **Region proposal network (RPN):** the first stage, which proposes candidate boxes; the ROI heads then classify, refine and mask each one.
- **Score and threshold:** the softmax class probability after non-maximum suppression; not calibrated. The threshold is the caller's decision about what is returned, not a model property.
- **NMS (non-maximum suppression):** drops overlapping boxes of the same class, keeping the highest-scoring one.
- **IoU:** intersection over union, for boxes or for masks.
- **AP50 / AP@[.50:.95]:** average precision with a match at IoU ≥ 0.50, and the mean over IoU thresholds 0.50–0.95; both rank every instance above the evaluation threshold by score.
- **Ceiling (saturated metric):** a metric at its maximum (AP 1.0); it cannot show a further gain, so it cannot rank two settings.
- **Re-heading:** replacing the class-specific box and mask predictors with freshly initialised ones for a new vocabulary.
- **Frozen backbone:** the ResNet-50 body keeps its COCO weights and BatchNorm statistics during fine-tuning.
- **Held-out split:** images never shown to the optimizer and never used to choose a setting; the task evidence.
- **Adapter / reload equivalence:** the SafeTensors file of the tensors the fine-tune could change, and the check that a fresh base model with the adapter loaded returns the same instances.
- **Isolated environment:** the separate hash-locked Python environment built in Section 1; every later cell runs there.

## Conclusion (your notes)

Optional. Fill in from your own run, one sentence each:

1. The pretrained COCO model matched ___ of 4 drawn objects at 0.75; the noise probe returned ___ instances at 0.05.
2. The fine-tune moved held-out mask AP from ___ to ___ and box `ap` from ___ to ___; the rows at the ceiling were ___.
3. What a 1.0 row on this held-out split does not tell me: ___.
4. In Section 14, unfreezing the backbone changed box `ap` by ___ and the training time by ___.
5. What I would need before using this adapter on real images: ___.

## References

- He, K., Gkioxari, G., Dollár, P. and Girshick, R. (2017). *Mask R-CNN.* [arXiv:1703.06870](https://arxiv.org/abs/1703.06870).
- Li, Y., Xie, S., Chen, X., Dollár, P., He, K. and Girshick, R. (2021). *Benchmarking Detection Transfer Learning with Vision Transformers.* [arXiv:2111.11429](https://arxiv.org/abs/2111.11429) — the paper torchvision cites for the v2 builder.
- torchvision model documentation: [maskrcnn_resnet50_fpn_v2](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn_v2.html); upstream repository [pytorch/vision](https://github.com/pytorch/vision) — BSD-3-Clause.
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common Objects in Context.* [arXiv:1405.0312](https://arxiv.org/abs/1405.0312).
- Repository model card: https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/maskrcnn-instance-segmentation-pipeline`](https://github.com/kurtvalcorza/maskrcnn-instance-segmentation-pipeline) — source repository for this pipeline.